# ARC-AGI-2 hybrid v2: NVARC floor, then Qwen3.8-27B verified program synthesis

**Stage A (floor)**: the team's NVARC v16 pipeline (Qwen3-4B + per-task LoRA TTT), cheap-first, capped at 6.5 h.
**Stage B (adds only)**: Qwen3.8-27B-FP8 on vLLM 0.19.0 (isolated CPython 3.12, installed on CPU while stage A
runs) writes `transform(grid)` programs; the harness runs them on every demo and gives pass/fail feedback.
v2: serving ladder (TP4 + MTP-1 speculative decoding -> TP4 -> the v1 flags), each checked by a sanity question;
fair-share token budgets from live throughput; low-effort first turns with execution-feedback repairs; a second
pass on unverified puzzles; the arckit helper toolkit preloaded in the sandbox.
**Merge**: attempt_1 is always NVARC's top-1; a demo-verified program output takes attempt_2.
**Diagnostic game**: every stage is a timed level graded PASS/WARN/FAIL; every puzzle prints its play line.
Fallback: if no layout starts (or all die), NVARC continues on the tasks it did not reach and Qwen's results are re-merged.


In [ ]:
# ---------------------------------------------------------------------------
# Clock, budgets, safety net, background vLLM install (CPU only - never touches the GPUs)
# ---------------------------------------------------------------------------
import os, time, json, random, glob, subprocess, sys, shutil
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
T0 = time.time()
HARD_END = T0 + 12 * 3600 - 1500          # Qwen generation / fallback stop; leaves 25 min for teardown + writes
QWEN_XHIGH_SOLVED = set(["0934a4d8", "136b0064", "16de56c4", "1ae2feb7", "20270e3b", "20a9e565", "2ba387bc", "2d0172a1", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4a21e3da", "581f7754", "58f5dbd5", "65b59efc", "67e490f4", "7491f3cf", "7666fa5d", "7b3084d4", "7c66cb00", "80a900e0", "898e7135", "8e5c0c38", "8f215267", "981571dc", "a6f40cea", "b5ca7ac4", "b99e7126", "bf45cf4b", "cb2d8a2c", "d35bdbdc", "d59b0160", "d8e07eb2", "db0c5428", "dbff022c", "e376de54", "e8686506", "edb79dae", "f931b4a8"])  # ARC Prize's verified per-task results (public eval) - diagnostics only
if RERUN:
    NVARC_END = T0 + 6.5 * 3600
    QWEN_WINDOW_S = 10 ** 9               # until HARD_END
    VAL_KEYS = None
else:
    # Validation: K public-eval tasks with exactly the per-task time a 240-task rerun gets.
    K = int(os.getenv("ARC_VAL_K", "40"))
    _ev = json.load(open("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"))
    VAL_KEYS = sorted(random.Random(0).sample(sorted(_ev), K))
    os.environ["ARC_DEBUG_KEYS"] = ",".join(VAL_KEYS)
    NVARC_END = T0 + 300 + K * (6.5 * 3600 - 300) / 240     # same NVARC seconds per task as the rerun
    QWEN_WINDOW_S = K * (12 * 3600 - 1500 - 6.5 * 3600 - 900) / 240   # same Qwen seconds per task
    HARD_END = NVARC_END + 1800 + QWEN_WINDOW_S + 600       # bounded: a failure cannot burn hours of quota
global_end_time = NVARC_END               # name used by the team's starter cells
DIAG = {"levels": [], "facts": {"rerun": RERUN, "val_keys": VAL_KEYS}}

def level(n, name, status, detail, t0):
    DIAG["levels"].append({"level": n, "name": name, "status": status, "s": round(time.time() - t0, 1), "detail": detail})
    print(f"<<< LEVEL {n} {name}: {status} in {time.time() - t0:.0f}s | {detail}", flush=True)
    json.dump(DIAG, open("/kaggle/working/diag_report.json", "w"), indent=1, default=str)

print(f"rerun={RERUN} NVARC stage {(NVARC_END - T0) / 3600:.2f} h; Qwen window "
      f"{'to the end' if RERUN else f'{QWEN_WINDOW_S / 60:.0f} min'}; val tasks {len(VAL_KEYS) if VAL_KEYS else 'all'}")

os.environ.setdefault("ARC_RESUME", "1" if RERUN else "0")
if os.environ["ARC_RESUME"] != "1":
    for _d in ("/kaggle/inference_outputs", "/kaggle/inference_outputs_deep"):
        shutil.rmtree(_d, ignore_errors=True)

# SAFETY NET: a schema-valid submission.json from the first seconds.
_sn_path = ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN
            else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
_challenges = json.load(open(_sn_path))
def _sn_attempts(inp):
    a1 = [[int(x) for x in row] for row in inp]
    a2 = [row[::-1] for row in a1]
    if a2 == a1:
        a2 = [row[:] for row in a1[::-1]]
    if a2 == a1:
        a2 = [[0]]
    return a1, a2
_sn_sub = {k: [dict(zip(("attempt_1", "attempt_2"), _sn_attempts(t["input"]))) for t in v["test"]] for k, v in _challenges.items()}
json.dump(_sn_sub, open("/kaggle/working/submission.json", "w"))
print(f"[safety-net] submission.json written: {len(_sn_sub)} tasks")

# Background CPU job: CPython 3.12 + offline vLLM 0.19.0 into /tmp/py312 (isolated from NVARC's python 3.11),
# then warm the page cache with the Qwen3.8 weights.
PREP = r"""
import glob, json, os, shutil, subprocess, tarfile, time
t = time.time(); st = {"ok": False}
try:
    tgz = (glob.glob("/kaggle/input/**/cpython312.tgz.bin", recursive=True) + glob.glob("/kaggle/input/**/cpython312.tar.gz", recursive=True))[0]
    shutil.rmtree("/tmp/py312", ignore_errors=True); os.makedirs("/tmp/py312")
    with tarfile.open(tgz, mode="r:gz") as tf:
        tf.extractall("/tmp/py312")
    py = "/tmp/py312/python/bin/python3.12"
    wh = os.path.dirname(glob.glob("/kaggle/input/**/vllm-0.19.0*.whl", recursive=True)[0])
    env = {"PATH": "/tmp/py312/python/bin:/usr/bin:/bin", "HOME": "/tmp", "TMPDIR": "/tmp", "PIP_NO_CACHE_DIR": "1"}
    r = subprocess.run([py, "-m", "pip", "install", "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
                        "--find-links", wh, "vllm==0.19.0"], env=env, capture_output=True, text=True, timeout=3000)
    st["pip_rc"] = r.returncode; st["pip_s"] = round(time.time() - t); st["pip_tail"] = (r.stdout + r.stderr)[-2500:]
    r2 = subprocess.run([py, "-c", "import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__)"],
                        env=env, capture_output=True, text=True, timeout=900)
    st["import"] = (r2.stdout + r2.stderr).strip()[-800:]
    st["ok"] = r.returncode == 0 and r2.returncode == 0
    st["python"] = py
except Exception as e:
    st["error"] = repr(e)
st["s"] = round(time.time() - t, 1)
json.dump(st, open("/kaggle/working/vllm_prep.json", "w"), indent=1)
for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
    try:
        if "Qwen3_5" in open(c).read():
            for f in sorted(glob.glob(os.path.join(os.path.dirname(c), "*.safetensors"))):
                subprocess.run(["cat", f], stdout=subprocess.DEVNULL, timeout=1200)
            break
    except Exception:
        pass
st["warm_s"] = round(time.time() - t, 1)
json.dump(st, open("/kaggle/working/vllm_prep.json", "w"), indent=1)
"""
open("/kaggle/working/prep_vllm.py", "w").write(PREP)
for _f in ("/kaggle/working/vllm_prep.json",):
    if os.path.exists(_f):
        os.remove(_f)
PREP_PROC = subprocess.Popen(["nice", "-n", "10", sys.executable, "/kaggle/working/prep_vllm.py"],
                             stdout=open("/kaggle/working/vllm_prep.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
print("[prep] background vLLM install started (CPU)")


In [ ]:
!pip uninstall -y tensorflow
# No torch.cuda calls in the kernel: a CUDA context here would hold GPU memory that vLLM needs later.
import os, sys, glob, subprocess, time
_t = time.time()
try:
    _smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True, timeout=60).stdout.strip().splitlines()
except Exception as _e:
    _smi = []
    print("nvidia-smi failed:", repr(_e))
print(sys.version.split()[0], "| GPUs:", _smi)
os.makedirs("/kaggle/working/logs", exist_ok=True)
_cfg_hits = glob.glob("/kaggle/input/models/sorokin/**/config.json", recursive=True)
print("NVARC model config:", _cfg_hits[:2] if _cfg_hits else "NONE - MODEL NOT ATTACHED!")
DIAG["facts"]["gpus"] = _smi
level(1, "hardware", "PASS" if len(_smi) == 4 and _cfg_hits else "WARN", f"{len(_smi)} GPU(s) {_smi[:1]}; NVARC model {'ok' if _cfg_hits else 'MISSING'}", _t)
if not _smi:  # no assert: an exception would void the run's submission; the safety net stays in place
    print("WARNING: no GPU visible - enable GPU before running!")


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))

# ---- v15: verified-transform solver (CPU, zero GPU cost) ----
# Only used as a SMART FALLBACK for test keys the LLM produced ZERO decoded
# candidates for (previously silently got the blind [[0]] guess). A candidate
# is emitted ONLY if it reproduces EVERY train pair exactly (verified), so it
# can never touch or downgrade a key that already has real decoder output --
# fill_submission() always overwrites these defaults when results exist.
# Isolated, additive change on top of the proven v10 (30.42) baseline -- the
# adaptive-budget / deterministic-seed / silent-fallback changes from the
# v11-v14 branches (28.19 / 29.31) are intentionally NOT included here, since
# those regressed score.
def _vt_geoms(a):
    return {
        "identity": a,
        "fliph": a[:, ::-1],
        "flipv": a[::-1, :],
        "rot90": np.rot90(a, 1),
        "rot180": np.rot90(a, 2),
        "rot270": np.rot90(a, 3),
        "transpose": a.T,
        "antitranspose": a[::-1, ::-1].T,
    }

def _vt_colormap(pairs):
    m = {}
    for i, o in pairs:
        if i.shape != o.shape:
            return None
        for a, b in zip(i.flatten(), o.flatten()):
            if a in m and m[a] != b:
                return None
            m[a] = b
    return m

def verified_transform_solve(train, test_input):
    try:
        pairs = [(np.array(p["input"]), np.array(p["output"])) for p in train]
        ti = np.array(test_input)
    except Exception:
        return None
    if not pairs:
        return None
    # 1) single geometric op consistent across all train pairs
    for name in ["identity", "fliph", "flipv", "rot90", "rot180", "rot270", "transpose", "antitranspose"]:
        ok = True
        for i, o in pairs:
            gi = _vt_geoms(i)[name]
            if gi.shape != o.shape or not np.array_equal(gi, o):
                ok = False
                break
        if ok:
            return _vt_geoms(ti)[name].tolist()
    # 2) global color remap
    m = _vt_colormap(pairs)
    if m is not None and all(np.array_equal(np.vectorize(lambda x: m.get(x, x))(i), o) for i, o in pairs):
        return np.vectorize(lambda x: m.get(x, x))(ti).tolist()
    # 3) integer tiling / pixel upscale with a constant ratio
    ratios = set()
    ok = True
    for i, o in pairs:
        if i.shape[0] == 0 or i.shape[1] == 0 or o.shape[0] % i.shape[0] or o.shape[1] % i.shape[1]:
            ok = False
            break
        ratios.add((o.shape[0] // i.shape[0], o.shape[1] // i.shape[1]))
    if ok and len(ratios) == 1:
        fr, fc = ratios.pop()
        if fr * fc > 1:
            for fn in [lambda g: np.tile(g, (fr, fc)),
                       lambda g: np.repeat(np.repeat(g, fr, axis=0), fc, axis=1)]:
                if all(fn(i).shape == o.shape and np.array_equal(fn(i), o) for i, o in pairs):
                    return fn(ti).tolist()
    # 4) constant output (every train output identical)
    outs = [o for _, o in pairs]
    if all(np.array_equal(outs[0], o) for o in outs):
        return outs[0].tolist()
    return None



class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def _fallback_attempts(self, k, i):
        # v15: smart fallback for test keys the LLM decoded ZERO candidates for
        # (was blind [[0]] before). fill_submission() always overwrites this
        # when real decoder results exist, so this can only help, never hurt.
        test_in = self.queries[k]['test'][i]['input']
        train = self.queries[k]['train']
        try:
            sol = verified_transform_solve(train, test_in)
        except Exception:
            sol = None
        if sol is not None:
            a1 = sol
            a2 = test_in                                  # diverse 2nd guess
        else:
            a1 = test_in                                  # ~67% of tasks are same-shape
            a2 = np.array(test_in)[::-1, :].tolist()      # flipped: cheap, distinct
        if a2 == a1:
            a2 = np.array(test_in)[:, ::-1].tolist()
        if a2 == a1:
            a2 = [[0]] if a1 != [[0]] else [[1]]
        return a1, a2
    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {}
        for k in self.keys:
            outs = []
            for i in range(len(self.queries[k]['test'])):
                a1, a2 = self._fallback_attempts(k, i)
                outs.append({'attempt_1': a1, 'attempt_2': a2})
            submission[k] = outs
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    # v16: primary key = selection score; exact ties fall back to the lowest (best) beam score seen
    # for that grid, so the ranking no longer depends on directory-listing order.
    scored = [(getter(sc), -min(g["beam_score"] for g in sc), o) for sc, o in scores.values()]
    scored = sorted(scored, key=(lambda x: (x[0], x[1])), reverse=True)
    ordered_outputs = [x[-1] for x in scored]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


def _valid_sample(sample):
    try:
        sol = np.asarray(sample["solution"])
        if sol.ndim != 2 or sol.size == 0 or sol.shape[0] > 30 or sol.shape[1] > 30:
            return False
        if not np.issubdtype(sol.dtype, np.integer):
            return False
        if sol.min() < 0 or sol.max() > 9:
            return False
        if not np.isfinite(sample["beam_score"]):
            return False
        if not len(sample["score_aug"]) or not np.all(np.isfinite(sample["score_aug"])):
            return False
        return True
    except Exception:
        return False


class ArcDecoder:

    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}
        try:
            self.valid_keys = set(dataset.keys)
        except Exception:
            self.valid_keys = None

    def load_decoded_results(self, store, run_name=""):
        if not os.path.isdir(store):
            print(f"*** No decoded results at {store}")
            return 0
        n_files = n_samples = n_bad = 0
        for key in sorted(os.listdir(store)):
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as e:
                print(f"*** Skipping corrupt shard {key}: {e}")
                n_bad += 1
                continue
            n_files += 1
            base_key = key.split(".")[0]
            if self.valid_keys is not None and base_key not in self.valid_keys:
                n_bad += 1
                continue
            for i, sample in enumerate(outputs):
                if not _valid_sample(sample):
                    n_bad += 1
                    continue
                self.decoded_results.setdefault(base_key, {})[f"{key}{run_name}.out{i}"] = sample
                n_samples += 1
        print(f"*** Loaded {n_files} shards / {n_samples} samples from {store} (skipped {n_bad})")
        return n_samples

    def candidate_stats(self):
        """Per output key: number of unique candidate grids and number of samples."""
        stats = {}
        for bk, v in self.decoded_results.items():
            uniq = {hashable(g["solution"]) for g in v.values()}
            stats[bk] = dict(unique=len(uniq), samples=len(v))
        return stats

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []
        correct_aug_means = []
        covered_outputs = set()

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                    correct_aug_means.append(aug_mean)
                    covered_outputs.add(basekey)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
            print(f" avg correct aug NLL:    {np.mean(correct_aug_means):8.5f}   (lower = model more confident in the right grid)")
        print(f" oracle coverage: {len(covered_outputs)}/{len(self.decoded_results)} outputs have the correct grid among decoded candidates")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import zlib
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle
import traceback

logging.disable(logging.WARNING)


import sys as _sys
_sys.setrecursionlimit(5000)   # turbo_dfs recurses once per output token (~931 deep)

# ---------------------------------------------------------------------------
# Run configuration (all overridable through environment variables so the
# same solver can run the LB-proven primary pass and a wider secondary pass).
# Defaults == the public LB 33.89 NVARC configuration.
# ---------------------------------------------------------------------------
def _env(name, default, cast):
    v = os.getenv(name)
    return cast(v) if v not in (None, "") else default

CFG = dict(
    model_path      = _env("ARC_MODEL_PATH", "", str),
    out_dir         = _env("ARC_OUT_DIR", "/kaggle/inference_outputs", str),
    lora_seed       = _env("ARC_LORA_SEED", 42, int),
    train_aug_seed  = _env("ARC_TRAIN_AUG_SEED", 1, int),
    n_train_aug     = _env("ARC_N_TRAIN_AUG", 16, int),     # x8 geometries = 128 sequences
    num_epochs      = _env("ARC_EPOCHS", 1, int),
    learning_rate   = _env("ARC_LR", 5e-5, float),
    eval_aug_seed   = _env("ARC_EVAL_AUG_SEED", 2, int),
    n_eval_aug      = _env("ARC_N_EVAL_AUG", 2, int),       # x8 geometries = 16 decoded views
    min_prob        = _env("ARC_MIN_PROB", 0.2, float),     # DFS cumulative prob threshold
    dfs_window      = _env("ARC_DFS_WINDOW", 540.0, float), # seconds per DFS call
    task_cap        = _env("ARC_TASK_CAP", 1200.0, float),  # seconds per task (decode phase)
    score_seed_off  = _env("ARC_SCORE_SEED_OFFSET", 0, int),
    decode_batch    = _env("ARC_DECODE_BATCH", 4, int),
)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "ÄŠ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


def resolve_model_dir():
    if CFG["model_path"] and os.path.isdir(CFG["model_path"]):
        return CFG["model_path"]
    candidates = [
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, "config.json")):
            return c
    import glob
    for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(c)
        if "grids15" in d and os.path.isfile(os.path.join(d, "tokenizer.json")):
            return d
    raise FileNotFoundError("qwen3_4b_grids15_sft139 model directory not found under /kaggle/input")


def stable_seed(key, offset=0):
    return (zlib.crc32(key.encode("utf-8")) + offset) % (1024 ** 2)


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Throughput patch (algebraically identical to the baseline): only the 12
# ARC-token NLL values are moved to the CPU instead of the whole vocabulary.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, dfs_window) -> dict:

    n = logits.size(0)

    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0])

    while time.time() - start_time < dfs_window and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
            dfs_window=dfs_window,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, dfs_window):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
        dfs_window=dfs_window,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def make_view_batches(eval_ds, n_perm, batch_size):
    """Group decoded views into DFS batches.

    Views inside one batch MUST share a prompt shape: rotating a non-square grid by 90/270 degrees changes
    the token count, and torch.tensor() of ragged prompts raises. The eight sorted geometries are
        0 id, 1 rot90, 2 rot180, 3 rot270, 4 T, 5 T.rot90, 6 T.rot180, 7 T.rot270
    and (id, rot180, T.rot90, T.rot270) keep the original grid shape while (rot90, rot270, T, T.rot180)
    swap it. Each pair below therefore stays inside one shape class. For n_perm=2 / batch_size=4 this is
    byte-for-byte the baseline batching ([id,rot180], [rot90,rot270], [T,T.rot180], [T.rot90,T.rot270]);
    for other n_perm (deep pass, n_perm=3) long lists are chunked WITHIN a pair instead of across pairs.
    """
    test_id_to_subkeys = defaultdict(list)
    for subkey in sorted(eval_ds.keys):
        test_id = subkey.split(".")[0].split("_")[1]
        test_id_to_subkeys[test_id].append(subkey)
    pairs_a = [(0, 2), (1, 3)]       # untransposed: (id, rot180), (rot90, rot270)
    pairs_b = [(4, 6), (5, 7)]       # transposed:   (T, T.rot180), (T.rot90, T.rot270)
    batches = []
    for pairs in (pairs_a, pairs_b):
        for test_id, subkeys in test_id_to_subkeys.items():
            for g0, g1 in pairs:
                views = subkeys[g0*n_perm:(g0+1)*n_perm] + subkeys[g1*n_perm:(g1+1)*n_perm]
                for i in range(0, len(views), batch_size):
                    batches.append(views[i:i+batch_size])
    return batches


def worker(rank, queue, end_time, test_path=None):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=CFG["lora_seed"],
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=CFG["num_epochs"],
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=CFG["learning_rate"],
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=CFG["lora_seed"],
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model_dir = resolve_model_dir()
    print(f"[Rank {rank}] model dir: {model_dir}")
    print(f"[Rank {rank}] config: {CFG}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_dir,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(CFG["min_prob"])

    if test_path is None:
        if rerun_mode:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        else:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = CFG["out_dir"]
    os.makedirs(dir_outputs, exist_ok=True)

    while True:

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        # resume support (ARC_RESUME=1): skip puzzles that already have a shard for every test output.
        # Shards are named "<task>_<idx>.<augmentation ops>", so match on the "<task>_<idx>." prefix.
        try:
            if os.getenv("ARC_RESUME", "0") == "1":
                _n_out = len(arc_test_set.queries[key]["test"])
                _have = os.listdir(dir_outputs)
                if all(any(f.startswith(f"{key}_{i}.") for f in _have) for i in range(_n_out)):
                    print(f"[Rank {rank}] skip {key}: all {_n_out} outputs already have shards")
                    continue
        except Exception:
            pass

        start_time = time.time()

        try:
            torch.cuda.reset_peak_memory_stats()

            set_peft_model_state_dict(
                model,
                default_weights.copy(),
                adapter_name="default",
            )

            model = FastLanguageModel.for_training(model)

            puzzle_ds = arc_test_set.change_keys([key])

            train_ds = puzzle_ds.augment(n=CFG["n_train_aug"], shfl_keys=True, seed=CFG["train_aug_seed"])
            train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):

                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )

                stats = trainer.train()

                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

                del trainer

            model = FastLanguageModel.for_inference(model)

            gc.collect()
            torch.cuda.empty_cache()

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

            torch.cuda.reset_peak_memory_stats()

            print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

            puzzle_ds_multi = puzzle_ds.split_multi_replies()

            eval_ds = puzzle_ds_multi.augment(n=CFG["n_eval_aug"], seed=CFG["eval_aug_seed"])
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

            batches = make_view_batches(eval_ds, CFG["n_eval_aug"], CFG["decode_batch"])

            with torch.inference_mode():

                known_scores = {}

                for subkeys in batches:

                    spend_time = time.time() - start_time
                    if spend_time > CFG["task_cap"] or time.time() > end_time:
                        print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                        break

                    print(f"[Rank {rank}] decoding {subkeys}")

                    tokens = []
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        tokens.append(tokenizer.encode(data["input"]))

                    # v16: prompts inside one DFS batch must have identical length (ragged input crashes
                    # torch.tensor). cut_to_len may drop a different example per view, so split by length;
                    # the normal all-equal-length batch is decoded in one call exactly as before.
                    by_len = defaultdict(list)
                    for j_, t_ in enumerate(tokens):
                        by_len[len(t_)].append(j_)
                    dfs_result = []
                    for idxs_ in by_len.values():
                        if time.time() > end_time:
                            break
                        sub_res_ = inference_turbo_dfs(model, [tokens[j_] for j_ in idxs_], max_new_tokens, max_score, end_time, CFG["dfs_window"])
                        dfs_result.extend((idxs_[b_], beams_) for b_, beams_ in sub_res_)

                    for subkey_id, scored_beams in dfs_result:

                        subkey = subkeys[subkey_id]
                        bk = subkey.split(".")[0]
                        decoded_result = []

                        for beam_score, tokens_ in scored_beams:

                            array = formatter.convert_tokens_to_array(tokens_)
                            if array is None:
                                continue

                            solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                            grid_id = (bk, tuple(map(tuple, solution)))

                            if grid_id in known_scores:
                                augmented_scores = known_scores[grid_id]
                            else:
                                print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                                aug_dataset = ArcDataset(
                                    keys=[bk],
                                    queries={bk: puzzle_ds_multi.queries.get(bk)},
                                    replies={bk: [solution.tolist()]},
                                )
                                aug_dataset = aug_dataset.augment(seed=stable_seed(bk, CFG["score_seed_off"]))
                                aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                                aug_queries = []
                                aug_answers = []
                                for augmented_sample in aug_dataset.as_list(formatter):
                                    aug_queries.append(augmented_sample["input"])
                                    aug_answers.append(augmented_sample["reply"])
                                augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                                augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                                augmented_scores = augmented_scores1 + augmented_scores2
                                known_scores[grid_id] = augmented_scores

                            decoded_result.append({
                                "beam_score": beam_score,
                                "score_aug": augmented_scores,
                                "solution": solution,
                            })

                        if len(decoded_result):
                            with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                                pickle.dump(decoded_result, f)

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")

        except Exception as e:
            print(f"[Rank {rank}] ERROR on puzzle {key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            try:
                model = FastLanguageModel.for_inference(model)
            except Exception:
                pass
            gc.collect()
            torch.cuda.empty_cache()
            if isinstance(e, torch.cuda.OutOfMemoryError):
                torch.cuda.synchronize()

        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import sys
import time
import json
import torch
import argparse
import traceback
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time, test_path, marker_dir):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Serialize the unsloth import/patching across workers (baseline behaviour),
    # but never wait forever for a dead predecessor.
    if rank > 0:
        waited = 0
        while not os.path.exists(os.path.join(marker_dir, f"worker{rank-1}")) and waited < 900:
            time.sleep(5)
            waited += 5

    from arc_solver import worker

    with open(os.path.join(marker_dir, f"worker{rank}"), "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")

    attempts = 0
    while attempts < 2 and time.time() < end_time:
        attempts += 1
        try:
            worker(rank, queue, end_time, test_path=test_path)
            break
        except Exception as e:
            print(f"[Rank {rank}] worker crashed ({type(e).__name__}: {e}); attempt {attempts}")
            traceback.print_exc()
            try:
                import gc
                gc.collect()
                torch.cuda.empty_cache()
            except Exception:
                pass
            if attempts >= 2:
                print(f"[Rank {rank}] giving up.")

    print(f"[Rank {rank}] done!")


def estimated_work(task):
    """Rough cost proxy: tokens of all train pairs (TTT + context) plus an estimate of
    the decoded output sizes. Cheap tasks first => the deadline truncates the most
    expensive (and least likely to be solved) tail."""
    def ntok(g):
        return len(g) * (len(g[0]) + 1)
    train_tokens = sum(ntok(p["input"]) + ntok(p["output"]) for p in task["train"])
    ratios = [ntok(p["output"]) / max(1, ntok(p["input"])) for p in task["train"]]
    ratios.sort()
    ratio = ratios[len(ratios) // 2]
    test_tokens = sum(ntok(t["input"]) * (1 + ratio) for t in task["test"])
    return train_tokens * 16 + test_tokens * 8 * len(task["test"])


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    parser.add_argument("--keys-file", type=str, default="")
    parser.add_argument("--nprocs", type=int, default=0)
    parser.add_argument("--order", type=str, default="cheap", choices=["cheap", "sorted", "file"])
    parser.add_argument("--test-path", type=str, default="")
    parser.add_argument("--marker-dir", type=str, default="/kaggle/working/markers")
    args, _ = parser.parse_known_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if args.test_path:
        test_path = args.test_path
    elif rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    if args.keys_file:
        with open(args.keys_file) as f:
            keys = [k for k in json.load(f) if k in data]
    else:
        keys = sorted(data.keys())
        if not rerun_mode:
            debug_keys = os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")
            keys = [k for k in keys if k in debug_keys]

    if args.order == "cheap":
        keys = sorted(keys, key=lambda k: estimated_work(data[k]))
    elif args.order == "sorted":
        keys = sorted(keys)

    nprocs = args.nprocs or min(4, torch.cuda.device_count())
    os.makedirs(args.marker_dir, exist_ok=True)
    for f_ in os.listdir(args.marker_dir):
        try:
            os.remove(os.path.join(args.marker_dir, f_))
        except Exception:
            pass

    print(f"[starter] {len(keys)} tasks, {nprocs} workers, order={args.order}, "
          f"budget={(args.end_time - time.time())/60:.1f} min, test_path={test_path}")

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    try:
        mp.spawn(local_worker, args=(queue, args.end_time, test_path, args.marker_dir), nprocs=nprocs)
    except Exception as e:
        print(f"[starter] spawn finished with error: {type(e).__name__}: {e}")
        traceback.print_exc()
    print("[starter] finished.")


In [ ]:
%%writefile qwen_stage.py
"""Stage B of the hybrid (v2): Qwen3.8-27B writes transform(grid) programs, the harness verifies them on the
demos, and verified outputs are merged into the NVARC submission without ever removing NVARC's top-1.

v2 changes, each driven by the v1 validation log (2026-10-05):
  * Serving layout ladder. v1 ran one TP=4 server: vLLM disabled custom all-reduce ("not supported on more than
    two PCIe-only GPUs") and Mamba 'align' prefix caching had a 0% hit rate. v2 tries two TP=2 servers first
    (custom all-reduce works for 2 PCIe GPUs), with and without MTP speculative decoding, prefix caching off,
    and falls back to the v1 TP=4 layout. A layout is accepted only if every server answers a sanity request.
  * Fair-share budgets. v1 gave every episode a 16k first turn at ~11 tok/s per stream, so late episodes were
    starved (3-4k tokens). v2 sizes each episode from measured throughput x time left / puzzles left.
  * Effort. Truncated xhigh never finished thinking (76 forced commits in 40 episodes). v2 uses low effort first
    (repairs from execution feedback), then a second pass on near-misses if time remains.
  * Toolkit. arckit (25 helpers distilled from 151 verified programs) is preloaded in the sandbox; the prompt
    carries a short cheat sheet.

Runs as a child process of the notebook (python 3.11, stdlib + numpy/scipy for the sandbox):
    python qwen_stage.py <config.json>
Exit code: 0 = ran (results merged), 3 = no serving layout could start (parent falls back to NVARC catch-up).
"""
from __future__ import annotations

import ast
import asyncio
import json
import os
import re
import signal
import subprocess
import sys
import textwrap
import time
import urllib.request

CFG: dict = {}
T_START = time.time()


def log(*a):
    print(f"[qwen {time.time() - T_START:7.1f}s]", *a, flush=True)


# ============================================================================ grids & prompts
def fmt(g) -> str:
    return f"{len(g)}x{len(g[0])}\n" + "\n".join(" ".join(str(int(v)) for v in r) for r in g)


SYSTEM_BASE = """You are an expert at ARC-AGI puzzles and a precise Python programmer.
Each puzzle has training examples (input grid -> output grid) that all follow ONE hidden rule.
Find the rule and implement it as a Python function:

def transform(grid: list[list[int]]) -> list[list[int]]

Grids are lists of rows of integers 0-9 (colours). numpy (np) and scipy.ndimage may be used.
Your function is executed on every training example and on the test inputs; you will be told
which examples pass or fail. The rule must be general: never hard-code an output grid.
Work efficiently: settle the output-size law and the role of each colour first, then write the code.

Answer format: one line starting with RULE: that states the rule, then exactly one
```python code block that defines transform (imports inside the block)."""

TOOLKIT_DOC = """

These helpers are already defined when your code runs (no import needed); use them when they fit:
bg(g) -> int most frequent colour (write b = bg(g); do not shadow the name) | inb(g,r,c) | nbrs(g,r,c,diag=False) -> [(r,c)]
objects(g, bg=None, diag=True, by_color=False) -> [Obj]  connected non-bg objects in reading order.
  CHECK diag (8- vs 4-connectivity) and by_color against the examples: neither default is always right.
  Obj: .cells [(r,c)] .r0 .c0 .r1 .c1 .h .w .size .mask(h x w bool) .full(grid bool) .patch(h x w) .color .colors .border .solid .n_holes .center
crop(g, obj|mask|(r0,c0,r1,c1)) -> array | holes(mask) -> [mask] enclosed regions
sep_lines(g, color=None) -> (rows, cols) full-span lines | panels(g, color) -> [[(r0,c0,sub)]] split at separator lines (pass the separator colour)
d8(a) -> 8 orientations | canon(a|Obj, rot=True, flip=True) -> shape key | same_shape(a, b, rot, flip, scale)
find(g, patch, wildcard=None) -> [(r,c)] | symmetry_fill(g, unknown) -> array
ray(g, start, d, stop=None) -> [(r,c)] | line(g, a, b, color) in place | paste(g, patch, r, c, transparent=None, color=None) in place
slide(g, obj, d, bg=None, move=True) -> steps | flood(passable, start, diag=False) -> mask
dist(passable, sources, diag=False) -> array (sources: [(r,c)], one (r,c), or a bool mask; -1 = unreachable)
outline(mask, diag=True) -> mask | kron(mask, tile, bg=0) -> array | period(a, axis=1, tol=0) -> int
exact_cover(target_mask, pieces, rot=False, flip=False) -> [(i, mask, r, c)] | None   (pieces: Objs or cropped masks)
Helpers take lists or numpy arrays; transform may return a list of lists or a 2-D numpy int array."""


def system_prompt() -> str:
    return SYSTEM_BASE + (TOOLKIT_DOC if CFG.get("toolkit_path") else "")


def task_prompt(task: dict) -> str:
    parts = []
    for i, p in enumerate(task["train"], 1):
        parts.append(f"## Example {i}\nInput {fmt(p['input'])}\nOutput {fmt(p['output'])}")
    for i, t in enumerate(task["test"], 1):
        parts.append(f"## Test input {i}\n{fmt(t['input'])}")
    return "\n\n".join(parts) + "\n\nWrite the RULE line and the python transform function."


def est_tokens(text: str) -> int:
    return int(len(text) / 1.9) + 200


# Tolerant fences: ```python / ```python3 / ```Python / ```py / bare ``` / ~~~, indented, CRLF, and a final
# fence left open when the turn hit max_tokens.
FENCE = re.compile(r"(```|~~~)[ \t]*([A-Za-z][A-Za-z0-9_+-]*)?[ \t]*\n(.*?)(?:\1|\Z)", re.S)


def _defs_only(src: str) -> bool:
    try:
        t = ast.parse(src)
    except SyntaxError:
        return False
    return all(isinstance(n, (ast.FunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom, ast.Assign, ast.AnnAssign))
               or (isinstance(n, ast.Expr) and isinstance(getattr(n, "value", None), ast.Constant)) for n in t.body)


def _trim_to_parsable(src: str) -> str | None:
    """A block cut off mid-way: the longest prefix that parses and still defines transform."""
    lines = src.splitlines()
    for k in range(len(lines), max(0, len(lines) - 120), -1):  # bounded: each try is a full parse
        cand = "\n".join(lines[:k])
        try:
            t = ast.parse(cand)
        except SyntaxError:
            continue
        if any(isinstance(n, ast.FunctionDef) and n.name == "transform" for n in t.body) or \
                any(isinstance(n, ast.Assign) and any(getattr(x, "id", None) == "transform" for x in n.targets) for n in t.body):
            return cand
    return None


def extract(text: str) -> tuple[str | None, str]:
    """Last python block that defines transform (def or lambda); definition-only blocks elsewhere in the reply
    (helpers) are prepended. Returns (code or None, RULE text)."""
    text = (text or "").replace("\r\n", "\n")
    blocks = []
    for m in FENCE.finditer(text):
        lang, body = (m.group(2) or "").lower(), textwrap.dedent(m.group(3))
        if lang and not lang.startswith("py"):
            continue
        blocks.append(body)
    rule = ""
    # [ \t...] not \s: with re.M, \s would cross newlines and make long blank output quadratic (event-loop stall)
    mm = re.findall(r"^[ \t*#>_-]*RULE\**:\**[ \t]*(.+)", text, re.M) or re.findall(r"RULE:[ \t]*(.+)", text)
    if mm:
        rule = mm[-1].strip().strip("*").strip()[:400]
    idx = [i for i, b in enumerate(blocks) if re.search(r"^\s*def transform\b|^transform\s*=", b, re.M)]
    if not idx:
        return None, rule
    i = idx[-1]
    main = blocks[i]
    try:
        ast.parse(main)
    except SyntaxError:
        main = _trim_to_parsable(main)
        if main is None:
            return None, rule
    # Only helper blocks that define functions/classes: a data-only block (e.g. expected = [[...]]) would make the
    # program look hard-coded.
    extra = [b for j, b in enumerate(blocks) if j != i and _defs_only(b) and "def transform" not in b
             and re.search(r"^\s*(def|class)\s", b, re.M)]
    code = "\n\n".join(extra + [main]) if extra else main
    return code.strip(), rule


def valid_grid(g) -> bool:
    return (isinstance(g, list) and 0 < len(g) <= 30 and isinstance(g[0], list) and 0 < len(g[0]) <= 30
            and all(isinstance(r, list) and len(r) == len(g[0]) and all(isinstance(v, int) and 0 <= v <= 9 for v in r) for r in g))


# ============================================================================ sandbox
RUNNER = r'''
import json, sys
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))
    resource.setrlimit(resource.RLIMIT_CPU, (24, 25))
    resource.setrlimit(resource.RLIMIT_CORE, (0, 0))
except Exception:
    pass
import contextlib, linecache, signal, time, traceback, types
T0 = time.time()
payload = json.loads(sys.stdin.read())
out = sys.stdout
def emit(d):
    out.write("\n@@RESULT@@" + json.dumps(d) + "\n"); out.flush()
def tb():  # the innermost frames (the failing line), with source shown via linecache
    return traceback.format_exc(limit=-3)[-700:]
class InputTimeout(BaseException):  # BaseException: a program's own `except Exception` cannot swallow it
    pass
def _alarm(*a):
    raise InputTimeout("this input took longer than %ds" % PER_INPUT)
PER_INPUT = int(payload.get("per_input_s", 6))
HAS_TIMER = hasattr(signal, "setitimer")
if HAS_TIMER:
    signal.signal(signal.SIGALRM, _alarm)
ns = {"__name__": "__arc__"}
toolkit_error = None
with contextlib.redirect_stdout(sys.stderr):  # prints in the program must not corrupt the result line
    if payload.get("toolkit"):
        try:  # its own module: a program that rebinds bg/inb/Obj/holes cannot break the helpers' internals
            src = open(payload["toolkit"]).read()
            linecache.cache["<arckit>"] = (len(src), None, src.splitlines(True), "<arckit>")
            kit = types.ModuleType("arckit")
            exec(compile(src, "<arckit>", "exec"), kit.__dict__)
            sys.modules["arckit"] = kit
            ns.update({k: v for k, v in kit.__dict__.items() if not k.startswith("_")})
        except BaseException:
            toolkit_error = traceback.format_exc(limit=-1)[-300:]
    linecache.cache["<program>"] = (len(payload["code"]), None, payload["code"].splitlines(True), "<program>")
    res = []
    try:
        exec(compile(payload["code"], "<program>", "exec"), ns)
        fn = ns["transform"]
    except BaseException:
        emit({"fatal": tb(), "toolkit_error": toolkit_error}); sys.exit(0)
    def norm(r):
        if hasattr(r, "tolist"): r = r.tolist()
        return [[int(v) for v in (row.tolist() if hasattr(row, "tolist") else row)] for row in r]
    for g in payload["inputs"]:
        # Run-wide budget below the RLIMIT_CPU / wall limits: report the inputs already done instead of
        # losing every result to a kill.
        if time.time() - T0 > 28 or time.process_time() > 20:
            res.append({"error": "skipped: the program used up the sandbox time on earlier inputs"})
            continue
        try:
            if HAS_TIMER:
                signal.setitimer(signal.ITIMER_REAL, PER_INPUT)
            try:
                res.append({"grid": norm(fn([row[:] for row in g]))})
            finally:
                if HAS_TIMER:
                    signal.setitimer(signal.ITIMER_REAL, 0)
        except BaseException:
            res.append({"error": tb()})
emit({"results": res, "toolkit_error": toolkit_error})
'''
SANDBOX_SEM: asyncio.Semaphore | None = None


async def run_program(code: str, inputs: list, timeout: float = 40.0) -> list[dict]:
    limits = {"OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
              "PYTHONHASHSEED": "0", "CUDA_VISIBLE_DEVICES": ""}
    if os.name == "posix":  # Kaggle: minimal environment, own process group so a timeout kills everything
        env, kw = {"PATH": "/usr/bin:/bin", **limits}, {"start_new_session": True}
    else:  # local Windows tests need the parent's system variables
        env, kw = {**os.environ, **limits}, {}
    async with SANDBOX_SEM:
        p = await asyncio.create_subprocess_exec(sys.executable, "-I", "-c", RUNNER, stdin=asyncio.subprocess.PIPE,
                                                 stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE, env=env, **kw)
        try:
            out, _ = await asyncio.wait_for(p.communicate(json.dumps(
                {"code": code, "inputs": inputs, "toolkit": CFG.get("toolkit_path"),
                 "per_input_s": CFG.get("per_input_s", 6)}).encode()), timeout)
        except asyncio.TimeoutError:
            try:
                os.killpg(p.pid, signal.SIGKILL) if os.name == "posix" else p.kill()
            except Exception:
                pass
            return [{"error": f"TimeoutError: program exceeded {timeout:.0f}s"}] * len(inputs)
    try:
        line = [l for l in out.decode(errors="replace").splitlines() if l.startswith("@@RESULT@@")][-1]
        d = json.loads(line[len("@@RESULT@@"):])
    except Exception:
        return [{"error": "program crashed (no result; memory or CPU limit?)"}] * len(inputs)
    if d.get("toolkit_error") and not STATS.get("toolkit_error"):
        STATS["toolkit_error"] = d["toolkit_error"]
        log(f"WARNING: arckit failed to load in the sandbox: {d['toolkit_error']}")
    if "fatal" in d:
        return [{"error": d["fatal"]}] * len(inputs)
    return d["results"]


def why_invalid(g) -> str:
    if not isinstance(g, list) or not g:
        return "empty output"
    if not all(isinstance(r, list) and r for r in g):
        return "rows are not non-empty lists"
    if len({len(r) for r in g}) > 1:
        return f"ragged rows (lengths {sorted({len(r) for r in g})})"
    if len(g) > 30 or len(g[0]) > 30:
        return f"too large ({len(g)}x{len(g[0])}; max 30x30)"
    bad = sorted({v for r in g for v in r if not (isinstance(v, int) and 0 <= v <= 9)}, key=str)[:5]
    return f"values outside 0-9: {bad}" if bad else "invalid grid"


def feedback(task: dict, results: list[dict]) -> tuple[str, int, list, dict]:
    """Per-example verdicts for the repair turn. Small wrong outputs are shown in full with a diff map
    ('.' = right, else the expected digit). Also returns facts used to word the repair request."""
    n = len(task["train"])
    lines, passed = [], 0
    facts = {"wrong_size": 0, "errors": 0, "max_wrong_frac": 0.0}
    for i, (p, r) in enumerate(zip(task["train"], results[:n]), 1):
        g, exp = r.get("grid"), p["output"]
        H, W = len(exp), len(exp[0])
        if g is None:
            facts["errors"] += 1
            lines.append(f"Example {i}: ERROR\n" + "\n".join((r.get("error") or "").strip().splitlines()[-6:]))
        elif not valid_grid(g):
            facts["errors"] += 1
            lines.append(f"Example {i}: FAIL - output is not a valid grid: {why_invalid(g)}")
        elif g == exp:
            passed += 1
            lines.append(f"Example {i}: PASS")
        elif (len(g), len(g[0])) != (H, W):
            facts["wrong_size"] += 1
            facts["max_wrong_frac"] = 1.0
            msg = f"Example {i}: FAIL - wrong size: got {len(g)}x{len(g[0])}, expected {H}x{W}"
            if len(g) * len(g[0]) <= 100:
                msg += "\nYour output:\n" + "\n".join(" ".join(map(str, row)) for row in g)
            lines.append(msg)
        else:
            diff = [(r_, c, exp[r_][c], g[r_][c]) for r_ in range(H) for c in range(W) if exp[r_][c] != g[r_][c]]
            facts["max_wrong_frac"] = max(facts["max_wrong_frac"], len(diff) / (H * W))
            msg = f"Example {i}: FAIL - {len(diff)} of {H * W} cells wrong"
            if H * W <= 300:
                bad = {(a, b): e for a, b, e, _ in diff}
                msg += ("\nYour output:\n" + "\n".join(" ".join(map(str, row)) for row in g) +
                        "\nDiff ('.' = correct, digit = the expected colour where yours is wrong):\n" +
                        "\n".join(" ".join(str(bad[(a, b)]) if (a, b) in bad else "." for b in range(W)) for a in range(H)))
            else:
                msg += ": " + "; ".join(f"({a},{b}) expected {e} got {x}" for a, b, e, x in diff[:16])
            lines.append(msg)
    tests = results[n:]
    for i, r in enumerate(tests, 1):
        g = r.get("grid")
        if g is not None and valid_grid(g):
            lines.append(f"Test input {i}: ran, output {len(g)}x{len(g[0])}")
        elif g is not None:
            lines.append(f"Test input {i}: invalid output: {why_invalid(g)}")
        else:
            lines.append(f"Test input {i}: ERROR\n" + "\n".join((r.get("error") or "no output").strip().splitlines()[-4:]))
    outs = [r.get("grid") if r.get("grid") is not None and valid_grid(r.get("grid")) else None for r in tests]
    return "\n".join(lines), passed, outs, facts


def repair_request(passed: int, n: int, facts: dict, literal: bool, test_failed: bool) -> str:
    if literal:
        return ("REJECTED: the code embeds demo output grids. Compute every output from its input grid. Reply with "
                "2-4 sentences of diagnosis, then the RULE: line and the complete corrected ```python block.")
    if passed == n and test_failed:
        head = "Every example passes but a test input fails: make the program handle the test input(s) too."
    elif passed == 0 and (facts["wrong_size"] or facts["max_wrong_frac"] > 0.25):
        head = ("No example passes and the outputs are far off: the RULE is probably wrong. Re-derive it from the "
                "examples (sizes, colours, objects) rather than patching the code.")
    elif facts["errors"] and not facts["wrong_size"]:
        head = "The program crashes: fix the failing line (see the traceback) and keep the rule unless it is wrong."
    else:
        head = "Probably an implementation bug: keep the passing examples passing and fix the failing ones."
    return head + (" Reply with 2-4 sentences of diagnosis, then the RULE: line and the complete corrected "
                   "```python block.")


def literal_output_in_code(task: dict, code: str) -> bool:
    """True if the code embeds a demo output grid (>= 9 cells) as a literal: JSON-style or multi-line lists,
    tuples, trailing commas, or one string per row."""
    flat = re.sub(r"\s+", "", code)
    norm = re.sub(r",([\]\)])", r"\1", flat).replace("(", "[").replace(")", "]")
    digits = re.sub(r"[^0-9]", "", code)
    for p in task["train"]:
        o = p["output"]
        if len(o) * len(o[0]) < 9:
            continue
        if json.dumps(o).replace(" ", "") in norm:
            return True
        rows = ["".join(str(v) for v in r) for r in o]
        if len(o) >= 3 and len(o[0]) >= 3 and all(f'"{r}"' in flat or f"'{r}'" in flat for r in rows):
            return True
        if len(o) * len(o[0]) >= 25 and len(set("".join(rows))) >= 3 and "".join(rows) in digits:
            return True
    return False


# ============================================================================ streaming client
class ServerDown(RuntimeError):
    pass


async def post_json(port: int, path: str, payload: dict, timeout: float = 60.0) -> dict:
    """Small non-streaming JSON POST (used for /tokenize and /detokenize)."""
    async def go():
        try:
            reader, writer = await asyncio.open_connection("127.0.0.1", port)
        except OSError as e:
            raise ServerDown(f"connect: {e!r}") from e
        body = json.dumps(payload).encode()
        writer.write((f"POST {path} HTTP/1.1\r\nHost: 127.0.0.1\r\nContent-Type: application/json\r\n"
                      f"Content-Length: {len(body)}\r\nConnection: close\r\n\r\n").encode() + body)
        try:
            await writer.drain()
            data = await reader.read()
        except OSError as e:
            raise ServerDown(f"{path}: {e!r}") from e
        finally:
            writer.close()
        head, _, rest = data.partition(b"\r\n\r\n")
        status = int(head.split(b" ", 2)[1])
        if b"transfer-encoding: chunked" in head.lower():
            out = b""
            while rest:
                size_line, _, rest = rest.partition(b"\r\n")
                size = int(size_line.split(b";")[0] or b"0", 16)
                if size == 0:
                    break
                out, rest = out + rest[:size], rest[size + 2:]
            rest = out
        if status >= 500:
            raise ServerDown(f"{path} HTTP {status}")
        if status != 200:
            raise RuntimeError(f"{path} HTTP {status}: {rest[:200]!r}")
        return json.loads(rest)
    return await asyncio.wait_for(go(), timeout)


async def stream_chat(port: int, payload: dict, sink: dict, path: str = "/v1/chat/completions"):
    STREAMS[port] = STREAMS.get(port, 0) + 1
    if STREAMS[port] == 1:
        LAST_ACT[port] = time.time()  # a quiet server is not a hung one until a stream waits on it
    try:
        return await _stream_chat(port, payload, sink, path)
    finally:
        STREAMS[port] -= 1


async def _stream_chat(port: int, payload: dict, sink: dict, path: str):
    sink.update({"content": "", "reasoning": "", "finish": None, "tokens": 0})
    payload = {**payload, "stream": True, "stream_options": {"include_usage": True}}
    try:
        reader, writer = await asyncio.open_connection("127.0.0.1", port)
    except OSError as e:
        raise ServerDown(f"connect: {e!r}") from e
    body = json.dumps(payload).encode()
    writer.write((f"POST {path} HTTP/1.1\r\nHost: 127.0.0.1\r\nContent-Type: application/json\r\n"
                  f"Content-Length: {len(body)}\r\nConnection: close\r\n\r\n").encode() + body)
    try:
        await writer.drain()
        head = await reader.readuntil(b"\r\n\r\n")
        status = int(head.split(b" ", 2)[1])
        chunked = b"transfer-encoding: chunked" in head.lower()

        async def pieces():
            if chunked:
                while True:
                    size = int((await reader.readuntil(b"\r\n")).split(b";")[0], 16)
                    if size == 0:
                        return
                    data = await reader.readexactly(size)
                    await reader.readexactly(2)
                    yield data
            else:
                while True:
                    data = await reader.read(65536)
                    if not data:
                        return
                    yield data
        if status != 200:
            data = b"".join([x async for x in pieces()])
            if status >= 500:
                raise ServerDown(f"HTTP {status}: {data[:200]!r}")
            raise RuntimeError(f"HTTP {status}: {data[:300]!r}")
        buf = b""
        async for piece in pieces():
            LAST_ACT[port] = time.time()
            buf += piece
            while b"\n\n" in buf:
                ev, buf = buf.split(b"\n\n", 1)
                for line in ev.split(b"\n"):
                    if not line.startswith(b"data:"):
                        continue
                    d = line[5:].strip()
                    if d == b"[DONE]":
                        return sink
                    o = json.loads(d)
                    if o.get("usage"):
                        sink["tokens"] = o["usage"].get("completion_tokens", sink["tokens"])
                    for ch in o.get("choices") or []:
                        dl = ch.get("delta") or {}
                        c = dl.get("content") or ch.get("text") or ""  # chat delta, or raw completion text
                        r = dl.get("reasoning_content") or dl.get("reasoning") or ""
                        sink["content"] += c
                        sink["reasoning"] += r
                        LIVE["chars"] += len(c) + len(r)
                        if ch.get("finish_reason"):
                            sink["finish"] = ch["finish_reason"]
    except (OSError, asyncio.IncompleteReadError) as e:
        raise ServerDown(f"stream: {e!r}") from e
    finally:
        writer.close()
    return sink


# ============================================================================ throughput bookkeeping
STATS = {"episodes": 0, "verified": 0, "tokens": 0, "chars": 0, "turns": 0, "commits": 0, "errors": 0,
         "server_down": 0, "rate_tok": 0.0, "rate_s": 0.0, "second_pass": 0, "layout": None}
LIVE = {"chars": 0}     # characters streamed so far (counted as they arrive, so long turns count while running)
RATE_SAMPLES: list = []  # (time, LIVE chars), sampled every 30 s
ACTIVE: dict = {}       # port -> episodes in flight
LAST_ACT: dict = {}     # port -> time of the last streamed bytes (hung-server watchdog)
STREAMS: dict = {}      # port -> open streams
KILL_EV: dict = {}      # port -> asyncio.Event set by the watchdog (aborts that server's open streams)


def per_stream_rate() -> float:
    prior = CFG.get("prior_stream_rate", 10.0)
    return (STATS["rate_tok"] + prior * 300) / (STATS["rate_s"] + 300)


def episode_budget(left: float, remaining: int) -> int:
    """Fair share: the streams play `remaining` puzzles in `left` seconds, so each gets aggregate_rate * left /
    remaining tokens; with fewer puzzles than streams, one stream's rate over the time left is the ceiling."""
    share = min(aggregate_rate() * left / max(1, remaining), per_stream_rate() * left)
    return int(max(CFG.get("min_budget", 5000), min(CFG.get("episode_tokens", 48000), 0.85 * share)))


def chars_per_token() -> float:
    return STATS["chars"] / STATS["tokens"] if STATS["tokens"] > 5000 and STATS["chars"] > 0 else 3.3


def aggregate_rate() -> float:
    """Live tokens/s over the last 15 minutes (streamed characters / chars-per-token); prior = per-stream x streams."""
    prior = per_stream_rate() * CFG["concurrency"]
    now = time.time()
    old = [s for s in RATE_SAMPLES if now - s[0] <= 900]
    if not old or now - old[0][0] < 240 or now - CFG.get("game_start", now) < 300:
        return prior
    return max(0.3 * prior, (LIVE["chars"] - old[0][1]) / chars_per_token() / (now - old[0][0]))


# ============================================================================ the game: one episode per task
async def turn(port: int, messages, effort: str | None, cap: int, box_end: float, thinking: bool = True,
               raw_prompt: list | str | None = None) -> dict:
    """One streamed request. raw_prompt: a /v1/completions continuation of an already-rendered prompt."""
    left = box_end - time.time() - 15
    cap = int(min(cap, max(0, left) * per_stream_rate() * 0.9))
    sink: dict = {"content": "", "reasoning": "", "finish": None, "tokens": 0}
    if cap < 500:
        sink["finish"] = "no-time"
        STATS["no_time"] = STATS.get("no_time", 0) + 1
        if STATS["no_time"] <= 3:
            log(f"no-time turn: {left:.0f}s left in box, per-stream {per_stream_rate():.1f} tok/s, cap {cap}")
        return sink
    # Qwen3.8 model card: thinking temperature 1.0 / top_p 0.95 / top_k 20; non-thinking 0.7 / 0.8 / 20
    # (its presence_penalty 1.5 is left out: it penalises the repeated tokens code is made of).
    temp, top_p = (CFG.get("temp_thinking", 1.0), 0.95) if thinking else (CFG.get("temp_direct", 0.7), 0.8)
    if raw_prompt is not None:
        path = "/v1/completions"
        payload = {"model": "qwen", "prompt": raw_prompt, "max_tokens": cap, "temperature": temp, "top_p": top_p, "top_k": 20}
    else:
        path = "/v1/chat/completions"
        kw = {"enable_thinking": thinking}
        if thinking and effort:
            kw["reasoning_effort"] = effort
        payload = {"model": "qwen", "messages": messages, "max_tokens": cap, "temperature": temp, "top_p": top_p,
                   "top_k": 20, "chat_template_kwargs": kw}
    t = time.time()
    st = asyncio.ensure_future(stream_chat(port, payload, sink, path))
    ev = KILL_EV.get(port)
    waiter = asyncio.ensure_future(ev.wait()) if ev is not None else None
    ok = False  # only completed or timeboxed turns calibrate the per-stream rate (not aborted ones)
    try:
        done, _ = await asyncio.wait([st] + ([waiter] if waiter else []), timeout=max(5.0, box_end - time.time()),
                                     return_when=asyncio.FIRST_COMPLETED)
        if st in done:
            st.result()  # re-raises ServerDown / RuntimeError from the stream
            ok = True
        elif waiter is not None and waiter in done:
            st.cancel()
            raise ServerDown("watchdog: server declared hung")
        else:
            st.cancel()
            sink["finish"] = "timebox"
            ok = True
    except ServerDown:
        raise
    except RuntimeError as e:  # HTTP 4xx for this request only (e.g. context length): end the turn, keep the puzzle
        sink["finish"] = "http-error"
        sink["error"] = str(e)[:200]
    finally:
        if waiter is not None:
            waiter.cancel()
        if not st.done():
            st.cancel()
        nch = len(sink["content"]) + len(sink["reasoning"])
        used = sink["tokens"] or int(nch / chars_per_token())
        if sink["tokens"]:  # usage reported: calibrates the live chars -> tokens conversion
            STATS["chars"] += nch
        else:
            STATS["chars"] += int(used * chars_per_token())
        sink["tokens"] = used
        STATS["tokens"] += used
        STATS["turns"] += 1
        dt = time.time() - t
        if ok and used > 200 and dt > 5:
            STATS["rate_tok"] += used
            STATS["rate_s"] += dt
    return sink


CUT = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
COMMIT_ASK = ("Your thinking was cut off. Do not continue analysing or re-check examples. Implement the most promising "
              "rule so far - it will be run on every example and you will get the results to fix it. Reply with only "
              "the RULE: line and one complete ```python block.")
NOCODE: list = []     # tails of replies without a usable program (diagnostics)


async def commit(port: int, base: list, s: dict, effort: str, cap: int, box_end: float, prompt_tokens: int) -> tuple[dict, str]:
    """Force an answer out of a cut-off thinking turn.
    Preferred ('continue'): resume the SAME generation with the whole reasoning trace, closed by the Qwen3
    thinking-budget hand-off sentence and </think> (Qwen3 technical report), via /v1/completions on token ids:
    the chat-template ids of the original prompt from /tokenize + the ids of the trace. No text round trip, so the
    special tokens of the template are exact. Fallback ('commit'): a new thinking-off chat turn that sees the last
    15k characters of the analysis."""
    # Drop <|...|> control strings: re-tokenized, an image placeholder without image data can crash vLLM <= 0.19.1.
    reasoning = re.sub(r"<\|[^|<>\s]{1,40}\|>", "", s["reasoning"]).strip()
    if CFG.get("commit_mode", "continue") == "continue" and prompt_tokens + s["tokens"] + cap + 400 < CFG["max_model_len"]:
        try:
            ids = (await post_json(port, "/tokenize", {"model": "qwen", "messages": base, "add_generation_prompt": True,
                                                       "chat_template_kwargs": {"enable_thinking": True, "reasoning_effort": effort}}))["tokens"]
            think = (await post_json(port, "/tokenize", {"model": "qwen", "prompt": "<think>\n", "add_special_tokens": False}))["tokens"]
            # The template may put effort text after <think>: look for the tag near the end, not only at the end.
            opened = bool(think) and think[0] in ids[-256:]
            if not STATS.get("commit_boundary"):
                STATS["commit_boundary"] = "pending"
                try:  # log once how the rendered prompt ends (validation check of the template assumption)
                    end = (await post_json(port, "/detokenize", {"model": "qwen", "tokens": ids[-40:]}))["prompt"]
                    STATS["commit_boundary"] = end[-160:]
                    log(f"continuation prompt ends with {end[-160:]!r}; think already open: {opened}")
                except Exception as e:
                    STATS["commit_boundary"] = f"detokenize failed: {e!r}"[:160]
            tail = (await post_json(port, "/tokenize", {"model": "qwen", "prompt": ("" if opened else "<think>\n") + reasoning + CUT,
                                                        "add_special_tokens": False}))["tokens"]
            s2 = await turn(port, None, None, cap, box_end, thinking=False, raw_prompt=ids + tail)
            if s2["finish"] != "http-error":
                return s2, "continue"
            STATS["continue_fail"] = STATS.get("continue_fail", 0) + 1
            if STATS["continue_fail"] <= 3:
                log(f"continuation commit refused: {s2.get('error')}")
        except ServerDown:
            raise
        except Exception as e:
            STATS["continue_fail"] = STATS.get("continue_fail", 0) + 1
            if STATS["continue_fail"] <= 3:
                log(f"continuation commit unavailable ({e!r}); using the chat commit")
    cm = base + [{"role": "assistant", "content": "My analysis so far:\n" + reasoning[-15000:]},
                 {"role": "user", "content": COMMIT_ASK}]
    return await turn(port, cm, None, cap, box_end, thinking=False), "commit"


async def episode(port: int, tid: str, task: dict, box_end: float, budget: int, effort: str, glog: list,
                  tag: str = "") -> dict:
    """Turn 1 thinks (effort) and should end in code; if the thinking is cut off, a commit turn (thinking off)
    turns the analysis into code. Then repair turns driven by execution feedback (thinking off by default: on
    2026-10-05, 6k-token thinking repairs fixed 1 program in 52) until verified or out of budget."""
    user = task_prompt(task)
    sysp = system_prompt()
    base = [{"role": "system", "content": sysp}, {"role": "user", "content": user}]
    room = CFG["max_model_len"] - est_tokens(sysp + user) - 512
    commit_cap = CFG.get("commit_cap", 3000)
    repair_cap = CFG.get("repair_cap", 3000)
    repair_thinking = CFG.get("repair_thinking", False)
    best = {"passed": -1, "code": None, "tests": None, "rule": ""}
    best_ctx = None
    spent, step = 0, []
    messages = list(base)
    n = len(task["train"])
    for k in range(1 + CFG.get("max_repairs", 2)):
        is_first = k == 0
        thinking = is_first or repair_thinking
        if is_first:  # keep tokens and seconds back for the commit turn and one repair
            cap = min(max(budget // 2, budget - commit_cap - repair_cap), room - commit_cap - 2500,
                      int(budget * CFG.get("first_frac", 0.85)))
            reserve_s = (commit_cap + repair_cap) / max(1.0, per_stream_rate()) + 40
        else:
            cap = min(budget - spent, room - 2500, repair_cap)
            reserve_s = (commit_cap / max(1.0, per_stream_rate()) + 30) if thinking else 0
        # Never more than 30% of the box, but always enough for a minimal (~600-token) commit when possible.
        reserve_s = max(min(reserve_s, 0.3 * max(0.0, box_end - time.time())),
                        min(reserve_s, 15 + 600 / max(1.0, per_stream_rate())))
        if cap < 700:
            break
        label = (effort if is_first else ("low" if thinking else "direct"))
        s = await turn(port, messages, effort if is_first else "low", cap, box_end - reserve_s, thinking=thinking)
        spent += s["tokens"]
        code, rule = extract(s["content"])
        if code is None and s["finish"] == "stop" and s["reasoning"]:
            # A finished turn whose answer landed in the reasoning stream (parser quirk): try its last code
            # block. It only counts if it passes every demo and the test checks, like any other program.
            code, rule = extract(s["reasoning"][-12000:])
        if code is None and s["reasoning"] and s["finish"] in ("length", "timebox", "stop"):
            # Commit: force the answer from the analysis so far.
            s2, how = await commit(port, base, s, effort if is_first else "low",
                                   max(1500, min(commit_cap, budget - spent)), box_end, est_tokens(sysp + user))
            if s2["finish"] != "no-time":
                STATS["commits"] += 1
                STATS["commit_" + how] = STATS.get("commit_" + how, 0) + 1
            spent += s2["tokens"]
            code, rule = extract(s2["content"])
            if code is None:  # last resort: a draft program inside the analysis (verified like any other)
                code, rule = extract(s["reasoning"][-15000:])
            step.append(f"t{k + 1}:{label} {s['tokens'] // 100 / 10}k {s['finish']}->{how} {s2['tokens']}tok")
        else:
            s2 = None
            step.append(f"t{k + 1}:{label} {s['tokens'] // 100 / 10}k {s['finish']}")
        if code is None:
            step[-1] += " no-code"
            last = (s2 or s)
            if len(NOCODE) < 80:
                NOCODE.append({"tid": tid, "turn": k + 1, "finish": last["finish"], "tail": (last["content"] or last["reasoning"])[-600:]})
            if s["finish"] in ("no-time", "timebox") or (s2 is not None and s2["finish"] in ("no-time", "timebox")):
                break
            messages = base + [{"role": "assistant", "content": (last["content"] or "")[-3000:] or "(no answer)"},
                               {"role": "user", "content": "Your reply had no ```python block defining transform. Write the RULE: line and the complete code now."}]
            continue
        res = await run_program(code, [p["input"] for p in task["train"]] + [t["input"] for t in task["test"]])
        fb, passed, tests, facts = feedback(task, res)
        step[-1] += f" demos {passed}/{n}"
        literal = literal_output_in_code(task, code)
        test_failed = any(g is None for g in tests)
        if passed == n and not test_failed and not literal:
            # The verified program's own outputs, always (never an earlier, rejected program's).
            best = {"passed": passed, "code": code, "tests": tests, "rule": rule, "verified": True}
            step[-1] += " VERIFIED"
            break
        cur = {"passed": passed, "code": code, "tests": tests, "rule": rule}
        ctx = {"code": code, "rule": rule, "fb": fb, "facts": facts, "literal": literal, "test_failed": test_failed, "passed": passed}
        if not literal and (passed > best["passed"] or (passed == best["passed"] and best["code"] and len(code) < len(best["code"]))):
            best = cur
            best_ctx = ctx
        if passed == n:
            step[-1] += " (rejected: hard-coded)" if literal else " (rejected: test input failed)"
        # Repair from the best program so far, not from a regression (v1: 1/5 -> 0/5, 2/4 -> 0/4 ...).
        use = best_ctx if (best_ctx is not None and best_ctx["passed"] > passed) else ctx
        messages = base + [{"role": "assistant", "content": f"RULE: {use['rule']}\n```python\n{use['code']}\n```"},
                           {"role": "user", "content": "Execution results:\n" + use["fb"] + "\n\n" +
                            repair_request(use["passed"], n, use["facts"], use["literal"], use["test_failed"])}]
    glog.append(f"PUZZLE {tid}{tag} | srv {port} budget {budget // 1000}k | {' > '.join(step) or 'not started'} | "
                f"{'VERIFIED' if best.get('verified') else 'unverified'} | {spent} tok")
    return {**best, "tokens": spent, "steps": step}


# ============================================================================ merge (the only code that writes answers)
def merge(nvarc_sub: dict, pool: dict, qres: dict) -> tuple[dict, dict]:
    """attempt_1 = NVARC top-1 wherever NVARC produced a candidate; a demo-verified program output takes the
    attempt_2 slot (else NVARC top-2). Without NVARC candidates: verified, then best unverified, then the floor."""
    out, how = {}, {"nvarc_only": 0, "verified_added": 0, "verified_same_as_n1": 0, "qwen_only": 0, "fallback": 0}
    for tid, entries in nvarc_sub.items():
        rows = []
        for i, e in enumerate(entries):
            key = f"{tid}_{i}"
            p = pool.get(key) or {}
            top = [g for g in p.get("top", []) if valid_grid(g)]
            # A floor grid outside the ARC limits (e.g. > 30 per side) can never be right; replace it so one bad
            # row cannot fail check_invariant and silently drop every Qwen result.
            f1 = e["attempt_1"] if valid_grid(e.get("attempt_1")) else [[0]]
            f2 = e["attempt_2"] if valid_grid(e.get("attempt_2")) else [[0]]
            q = qres.get(tid) or {}
            qt = q.get("tests") or []
            V = qt[i] if (q.get("verified") and i < len(qt) and valid_grid(qt[i])) else None
            U = qt[i] if (not q.get("verified") and q.get("passed", 0) > 0 and i < len(qt) and valid_grid(qt[i])) else None
            if top:
                a1 = top[0]
                if V is not None and V != a1:
                    a2 = V
                    how["verified_added"] += 1
                else:
                    a2 = top[1] if len(top) > 1 else f2
                    how["verified_same_as_n1" if V is not None else "nvarc_only"] += 1
            else:
                cands = [g for g in (V, U) if g is not None]
                if cands:
                    a1 = cands[0]
                    a2 = cands[1] if len(cands) > 1 and cands[1] != a1 else f1
                    how["qwen_only"] += 1
                else:
                    a1, a2 = f1, f2
                    how["fallback"] += 1
            rows.append({"attempt_1": a1, "attempt_2": a2})
        out[tid] = rows
    return out, how


def check_invariant(sub: dict, pool: dict) -> bool:
    for tid, rows in sub.items():
        for i, r in enumerate(rows):
            top = [g for g in (pool.get(f"{tid}_{i}") or {}).get("top", []) if valid_grid(g)]
            if top and r["attempt_1"] != top[0]:
                return False
            if not (valid_grid(r["attempt_1"]) and valid_grid(r["attempt_2"])):
                return False
    return True


def atomic_write(path: str, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)


# ============================================================================ serving layouts
SERVERS: list = []   # [{"port", "proc", "log"}]


def server_env(gpus: str) -> dict:
    keep_ld = [p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p and "python3.11" not in p and "dist-packages" not in p]
    py = CFG["python312"]
    home = f"/tmp/vllmhome_{gpus.replace(',', '')}"
    for d in (home, home + "/tmp"):
        os.makedirs(d, exist_ok=True)
    return {"PATH": f"{os.path.dirname(py)}:/usr/local/cuda/bin:/usr/bin:/bin", "HOME": home,
            "LD_LIBRARY_PATH": ":".join(keep_ld + ["/usr/local/nvidia/lib64", "/usr/local/nvidia/lib"]),
            "LIBRARY_PATH": "/usr/local/nvidia/lib64:/usr/local/cuda/lib64/stubs",
            "CUDA_VISIBLE_DEVICES": gpus, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
            "VLLM_USE_FLASHINFER_SAMPLER": "0", "VLLM_NO_USAGE_STATS": "1", "PYTHONNOUSERSITE": "1",
            "TMPDIR": home + "/tmp", "XDG_CACHE_HOME": home + "/.cache", "TRITON_CACHE_DIR": home + "/triton",
            "TORCHINDUCTOR_CACHE_DIR": home + "/inductor", "VLLM_CACHE_ROOT": home + "/vllm",
            "TOKENIZERS_PARALLELISM": "false", "OMP_NUM_THREADS": "8"}


def sanity(port: int) -> bool:
    """A layout must answer a question correctly (catches a server that runs but decodes garbage)."""
    body = json.dumps({"model": "qwen", "messages": [{"role": "user", "content": "What is 17 + 25? Reply with only the number."}],
                       "max_tokens": 24, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}}).encode()
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=body,
                                     headers={"Content-Type": "application/json"})
        r = json.loads(urllib.request.urlopen(req, timeout=300).read())
        msg = r["choices"][0]["message"]
        text = (msg.get("content") or "") + (msg.get("reasoning_content") or msg.get("reasoning") or "")
        log(f"sanity on port {port}: {text.strip()[:60]!r}")
        return "42" in text
    except Exception as e:
        log(f"sanity request on port {port} failed: {e!r}")
        return False


def gpu_pids() -> list:
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout
        return [int(x) for x in out.split() if x.strip().isdigit()]
    except Exception:
        return []


def stop_servers():
    """SIGTERM each server's process group, wait, then always SIGKILL the group (EngineCore / TP workers can
    outlive the API server, e.g. a rank stuck in NCCL); finally kill leftover GPU holders that are verifiably
    vLLM processes (checked through /proc, in case NVML reports pids from another namespace)."""
    for s in SERVERS:
        p = s.get("proc")
        if p is None:
            continue
        pgid = p.pid
        try:
            os.killpg(pgid, signal.SIGTERM) if os.name == "posix" else p.kill()
        except Exception:
            pass
        try:
            p.wait(timeout=45)
        except Exception:
            pass
        if os.name == "posix":
            try:
                os.killpg(pgid, signal.SIGKILL)
            except Exception:
                pass
        s["proc"] = None
    if os.name == "posix" and not CFG.get("mock"):
        t = time.time()
        while time.time() - t < 120:
            pids = gpu_pids()
            if not pids:
                break
            if time.time() - t > 30:
                for pid in pids:
                    if is_vllm_pid(pid):
                        try:
                            os.kill(pid, signal.SIGKILL)
                        except Exception:
                            pass
            time.sleep(3)
        left = gpu_pids()
        if left:
            log(f"WARNING: GPU still held by pids {left} after stop_servers")


def is_vllm_pid(pid: int) -> bool:
    """Only processes that are verifiably ours (the CPython 3.12 that runs vLLM) may be killed by pid."""
    try:
        cmd = open(f"/proc/{pid}/cmdline", "rb").read().decode(errors="ignore")
        return bool(cmd) and (CFG.get("python312", "\0") in cmd or "vllm" in cmd)
    except Exception:
        return False


def has_mtp_weights(model_dir: str) -> bool:
    """vLLM silently skips safetensors files missing from the index, and an unloaded MTP head still passes the
    sanity question (it only slows decoding). Use an MTP layout only if the index lists mtp.* weights."""
    try:
        idx = json.load(open(os.path.join(model_dir, "model.safetensors.index.json")))
        keys = idx.get("weight_map", {})
        return any(k.startswith("mtp.") or ".mtp." in k for k in keys)
    except Exception as e:
        log(f"no readable safetensors index ({e!r}); assuming MTP weights are present")
        return True


def server_stats(s: dict) -> str:
    """Latest vLLM periodic metrics from the server log: generation throughput, running requests, and the
    speculative-decoding acceptance rate (MTP layouts)."""
    try:
        with open(s["log"], "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 60000))
            txt = f.read().decode(errors="ignore")
    except Exception:
        return ""
    gen = re.findall(r"Avg generation throughput: ([\d.]+) tokens/s, Running: (\d+) reqs", txt)
    acc = re.findall(r"Avg Draft acceptance rate: ([\d.]+)%", txt)
    parts = []
    if gen:
        parts.append(f"gen {gen[-1][0]} tok/s running {gen[-1][1]}")
    if acc:
        parts.append(f"MTP accept {acc[-1]}%")
    return f"{s['port']}: " + ", ".join(parts) if parts else ""


def start_layout(layout: dict) -> bool:
    """Start every server of a layout in parallel; accept only if all become ready and answer."""
    SERVERS.clear()
    if CFG.get("mock"):  # local tests: fake servers are already running
        if layout.get("mock_fail"):
            log(f"[{layout['name']}] (mock) start failed")
            return False
        SERVERS.extend({"port": s["port"], "proc": None, "max_seqs": s["max_seqs"], "conc": s["max_seqs"]} for s in layout["servers"])
        return True
    if any("--speculative-config" in s["flags"] for s in layout["servers"]) and not has_mtp_weights(CFG["model_dir"]):
        log(f"[{layout['name']}] skipped: the model index lists no mtp.* weights")
        return False
    for i, s in enumerate(layout["servers"]):
        flags = list(s["flags"])
        cmd = [CFG["python312"], "-m", "vllm.entrypoints.openai.api_server", "--model", CFG["model_dir"],
               "--served-model-name", "qwen", "--port", str(s["port"]), "--host", "127.0.0.1"] + flags
        logp = os.path.join(CFG["work_dir"], f"vllm_{layout['name']}_{i}.log")
        log(f"[{layout['name']}] starting server {i} on GPUs {s['gpus']}: {' '.join(flags)}")
        p = subprocess.Popen(cmd, env=server_env(s["gpus"]), stdout=open(logp, "w"), stderr=subprocess.STDOUT,
                             start_new_session=True)
        SERVERS.append({"port": s["port"], "proc": p, "log": logp, "max_seqs": s["max_seqs"]})
    t0 = time.time()
    limit = min(CFG.get("start_timeout_s", 1500), CFG["deadline"] - time.time() - 900)
    ready = set()
    while time.time() - t0 < limit and len(ready) < len(SERVERS):
        for s in SERVERS:
            if s["port"] in ready:
                continue
            if s["proc"].poll() is not None:
                log(f"[{layout['name']}] server on port {s['port']} exited ({s['proc'].poll()}):\n"
                    + open(s["log"], errors="ignore").read()[-2500:])
                stop_servers()
                return False
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{s['port']}/v1/models", timeout=5)
                ready.add(s["port"])
            except Exception:
                pass
        time.sleep(5)
    if len(ready) < len(SERVERS):
        log(f"[{layout['name']}] not ready after {time.time() - t0:.0f}s")
        stop_servers()
        return False
    for s in SERVERS:
        if not sanity(s["port"]):
            stop_servers()
            return False
        txt = open(s["log"], errors="ignore").read()
        kv = re.search(r"GPU KV cache size: ([\d,]+) tokens", txt)
        mx = re.search(r"Maximum concurrency for ([\d,]+) tokens per request: ([\d.]+)x", txt)
        # For this hybrid model the "KV cache size" line counts one layer group only (v1: 196,000 tokens while
        # the concurrency line said 11.12 x 65,536 = 729k). Capacity = the concurrency line. v1 used 32% of it at
        # 22 streams (~10.6k tokens per stream on average), so 16k per stream leaves room for peaks.
        cap_tokens = float(mx.group(2)) * int(mx.group(1).replace(",", "")) if mx else None
        s["conc"] = max(4, min(s["max_seqs"], int(cap_tokens / CFG.get("tokens_per_stream", 16000)))) if cap_tokens else s["max_seqs"]
        s["kv_tokens"] = int(kv.group(1).replace(",", "")) if kv else None
        s["kv_capacity"] = int(cap_tokens) if cap_tokens else None
        log(f"[{layout['name']}] port {s['port']} READY; KV capacity {s['kv_capacity']} tokens "
            f"(concurrency line {mx.group(2) if mx else '?'}x) -> {s['conc']} streams")
    log(f"[{layout['name']}] layout READY in {time.time() - t0:.0f}s")
    return True


# ============================================================================ scheduler
QRES: dict = {}       # tid -> best episode result (kept across layout restarts)
GLOG: list = []       # one PUZZLE line per episode
SECOND: set = set()   # puzzles queued for their second pass
PLAYED2: set = set()  # puzzles whose second pass finished
CONFIDENT: set = set()  # puzzles where every NVARC output has top-1 vote share >= 50%


def write_outputs(nvarc_sub: dict, pool: dict) -> dict:
    sub, how = merge(nvarc_sub, pool, QRES)
    if check_invariant(sub, pool):
        atomic_write(CFG["out_sub"], sub)
        atomic_write(os.path.join(CFG["work_dir"], "qwen_results.json"),
                     {"results": QRES, "stats": STATS, "merge": how, "game_log": GLOG[-600:], "nocode": NOCODE,
                      "servers": [{k: v for k, v in s.items() if k != "proc"} for s in SERVERS]})
    else:
        log("MERGE INVARIANT FAILED - submission not replaced")
    return how


async def run_all(tasks: dict, order: list, pool: dict, nvarc_sub: dict) -> str:
    """Play every puzzle once (cheap-first order), then replay unverified ones while time remains.
    Returns 'dead' if every server of the layout died (caller may start the next layout), else 'done'."""
    global SANDBOX_SEM
    SANDBOX_SEM = asyncio.Semaphore(12)
    deadline = CFG["deadline"]
    CFG.setdefault("game_start", time.time())
    stop_new = min(240.0, 0.15 * max(1.0, deadline - time.time()))
    queue = [(t, 1) for t in order if t not in QRES]
    SECOND.intersection_update(PLAYED2)  # replays lost with a dead layout are queued again
    in_flight = {"n": 0}
    last_write = [time.time()]
    fails = {s["port"]: 0 for s in SERVERS}
    errors: dict = {}
    state = {"dead": False}
    RATE_SAMPLES.clear()  # samples from a dead layout would include the restart gap
    for s in SERVERS:
        ACTIVE[s["port"]] = 0
        STREAMS[s["port"]] = 0
        LAST_ACT[s["port"]] = time.time()
        KILL_EV[s["port"]] = asyncio.Event()

    def pick_port() -> int:
        live = [s for s in SERVERS if s["conc"] > 0] or SERVERS
        return min(live, key=lambda x: ACTIVE[x["port"]] / max(1, x["conc"]))["port"]

    def refill_second_pass() -> int:
        """First-pass queue empty and time left: replay finished, unverified puzzles (near-misses first), so idle
        streams never wait for the slowest first-pass episodes. Each puzzle gets at most one replay."""
        if not CFG.get("second_pass", True) or deadline - time.time() < max(CFG.get("second_pass_min_left_s", 900), 3 * stop_new):
            return 0
        cands = [t for t, r in QRES.items() if t not in SECOND and not r.get("verified")]
        cands.sort(key=lambda t: (-(max(0, QRES[t].get("passed", -1)) / max(1, len(tasks[t]["train"]))),
                                  QRES[t].get("passed", -1) < 0, est_tokens(task_prompt(tasks[t]))))
        for t in cands:
            SECOND.add(t)
            queue.append((t, 2))
        return len(cands)

    def accepting() -> bool:
        # No new episode unless its box (which ends 30 s before the deadline) can hold the commit reserve
        # (15 s + 600 tokens) and a 500-token first turn with margin; otherwise episodes would end instantly as
        # 'no-time' and drain the queue without playing.
        return time.time() < deadline - max(stop_new, 90 + 1800 / max(1.0, per_stream_rate())) and not state["dead"]

    async def worker():
        while accepting():
            if not queue:
                if refill_second_pass():
                    continue
                if in_flight["n"] == 0:
                    return
                await asyncio.sleep(5)
                continue
            tid, pas = queue.pop(0)
            port = pick_port()
            ACTIVE[port] += 1
            in_flight["n"] += 1
            try:
                # NVARC-confident puzzles (queued last; NVARC was right on 5 of 7 in v1) do not dilute the budgets
                # of the others: they share whatever is left when they come up.
                nonconf = sum(1 for t, _ in queue if t not in CONFIDENT)
                remaining = (nonconf + in_flight["n"]) if (pas == 1 and tid not in CONFIDENT) else (len(queue) + in_flight["n"])
                budget = episode_budget(deadline - stop_new - time.time(), remaining)
                box_end = min(deadline - 30, time.time() + budget / max(1.0, per_stream_rate()) * 1.25 + 120)
                effort = CFG.get("effort_first", "low") if pas == 1 else CFG.get("effort_second", "low")
                r = await episode(port, tid, tasks[tid], box_end, budget, effort, GLOG, tag="" if pas == 1 else " [2nd]")
                STATS["episodes"] += 1
                if r.get("tokens", 0) > 0:  # the server actually answered (a no-time episode proves nothing)
                    fails[port] = 0
                r["pass"] = pas
                if pas == 2:
                    STATS["second_pass"] += 1
                    PLAYED2.add(tid)
                prev = QRES.get(tid)
                if prev is None or (r.get("verified") and not prev.get("verified")) or \
                        (not prev.get("verified") and r.get("passed", -1) > prev.get("passed", -1)):
                    QRES[tid] = r
                if r.get("verified"):
                    STATS["verified"] += 1
                print(GLOG[-1], flush=True)
            except ServerDown as e:
                STATS["server_down"] += 1
                fails[port] += 1
                log(f"server {port} down during {tid} (consecutive {fails[port]}): {e}")
                queue.insert(0, (tid, pas))
                for s in SERVERS:
                    exited = s.get("proc") is not None and s["proc"].poll() is not None
                    if s["conc"] > 0 and (exited or fails[s["port"]] >= 5):
                        s["conc"] = 0
                        log(f"server {s['port']} marked dead ({'exited' if exited else '5 consecutive failures'})")
                if all(s["conc"] == 0 for s in SERVERS):
                    log("no live server in this layout")
                    state["dead"] = True
                    return
                await asyncio.sleep(20)
            except Exception as e:  # unexpected (not the server): retry the puzzle once, then record a placeholder
                STATS["errors"] += 1
                errors[tid] = errors.get(tid, 0) + 1
                GLOG.append(f"PUZZLE {tid} | ERROR {e!r}"[:300])
                if errors[tid] == 1:
                    queue.append((tid, pas))
                elif tid not in QRES:
                    QRES[tid] = {"passed": -1, "code": None, "tests": None, "rule": "", "pass": pas, "error": repr(e)[:200]}
            finally:
                ACTIVE[port] -= 1
                in_flight["n"] -= 1
            if time.time() - last_write[0] > CFG.get("log_every_s", 300):
                last_write[0] = time.time()
                how = write_outputs(nvarc_sub, pool)
                el = time.time() - CFG["game_start"]
                srv = "; ".join(x for x in (server_stats(s) for s in SERVERS if s.get("log")) if x)
                log(f"HUD | layout {STATS['layout']} | episodes {STATS['episodes']} done {len(QRES)}/{len(tasks)} verified "
                    f"{sum(1 for r in QRES.values() if r.get('verified'))} (2nd pass {STATS['second_pass']}) | tokens "
                    f"{STATS['tokens'] / 1e6:.2f}M ({STATS['tokens'] / max(1, el):.0f} tok/s, recent {aggregate_rate():.0f}) | "
                    f"per-stream {per_stream_rate():.1f} | commits {STATS['commits']} | merge {how} | queue {len(queue)} | "
                    f"{(deadline - time.time()) / 60:.0f} min left" + (f" | vLLM {srv}" if srv else ""))

    async def sampler():
        """Throughput samples every 30 s, plus the hung-server watchdog: streams open but no bytes for
        hang_s seconds -> kill that server's process group, so its streams fail fast (ServerDown) and the
        layout ladder can move on instead of waiting out hour-long time boxes."""
        hang_s = CFG.get("hang_s", 420)
        while True:
            now = time.time()
            RATE_SAMPLES.append((now, LIVE["chars"]))
            del RATE_SAMPLES[:-60]
            for s in SERVERS:
                p = s.get("proc")
                if s["conc"] > 0 and STREAMS.get(s["port"], 0) > 0 and now - LAST_ACT.get(s["port"], now) > hang_s:
                    log(f"WATCHDOG: server {s['port']} streamed nothing for {now - LAST_ACT[s['port']]:.0f}s with "
                        f"{STREAMS[s['port']]} open streams - killing it")
                    s["conc"] = 0
                    KILL_EV[s["port"]].set()
                    if p is not None and os.name == "posix":
                        try:
                            os.killpg(p.pid, signal.SIGKILL)
                        except Exception:
                            pass
                    if all(x["conc"] == 0 for x in SERVERS):
                        state["dead"] = True
            await asyncio.sleep(30)

    samp = asyncio.create_task(sampler())
    workers = [asyncio.create_task(worker()) for _ in range(CFG["concurrency"])]
    try:
        await asyncio.wait_for(asyncio.gather(*workers), timeout=max(10, deadline - time.time()))
    except asyncio.TimeoutError:
        log("deadline: cancelling in-flight episodes")
    for w in workers + [samp]:
        w.cancel()
    await asyncio.sleep(0.5)
    write_outputs(nvarc_sub, pool)
    return "dead" if state["dead"] else "done"


def main():
    CFG.update(json.load(open(sys.argv[1])))
    CFG["deadline"] = CFG.get("hard_deadline", CFG.get("deadline"))
    CFG["max_model_len"] = CFG.get("max_model_len", 65536)
    tasks_all = json.load(open(CFG["test_path"]))
    tasks = {k: tasks_all[k] for k in CFG["keys"] if k in tasks_all}
    pool = json.load(open(CFG["nvarc_pool"]))
    nvarc_sub = json.load(open(CFG["nvarc_sub"]))

    def cls(tid):  # 0 = NVARC unsure, 1 = NVARC never decoded, 2 = NVARC confident
        rows = [pool.get(f"{tid}_{i}") for i in range(len(tasks[tid]["test"]))]
        if any(not r or not r.get("top") for r in rows):
            return 1
        return 2 if all(r.get("share", 0) >= 0.5 for r in rows) else 0
    order = sorted(tasks, key=lambda t: (cls(t) == 2, est_tokens(task_prompt(tasks[t]))))
    CONFIDENT.update(t for t in tasks if cls(t) == 2)
    log(f"{len(tasks)} tasks; classes unsure/undecoded/confident = "
        f"{sum(cls(t) == 0 for t in tasks)}/{sum(cls(t) == 1 for t in tasks)}/{sum(cls(t) == 2 for t in tasks)}; "
        f"hard deadline in {(CFG['deadline'] - time.time()) / 60:.1f} min")
    played, window_set, status = False, False, "none"
    layouts = list(CFG["layouts"])
    while layouts:
        layout = layouts.pop(0)
        if CFG["deadline"] - time.time() < CFG.get("min_start_left_s", 1800):
            log(f"too little time left to start [{layout['name']}]")
            break
        try:
            started = start_layout(layout)
        except Exception as e:  # never leave a half-started layout holding the GPUs
            log(f"[{layout['name']}] start raised {e!r}")
            stop_servers()
            started = False
        if not started:
            continue
        STATS["layout"] = layout["name"]
        STATS.setdefault("layouts_used", []).append(layout["name"])
        STATS.setdefault("layout_ready_s", []).append(round(time.time() - T_START))
        CFG["concurrency"] = sum(s["conc"] for s in SERVERS)
        if not window_set:
            # The game clock starts when the first layout is ready (validation runs get the rerun's seconds per task).
            CFG["deadline"] = min(CFG["deadline"], time.time() + CFG.get("window_s", 10 ** 9))
            window_set = True
        log(f"layout {layout['name']}: {len(SERVERS)} server(s), {CFG['concurrency']} streams; "
            f"{(CFG['deadline'] - time.time()) / 60:.1f} min left")
        played = True
        try:
            status = asyncio.run(run_all(tasks, order, pool, nvarc_sub))
        finally:
            stop_servers()
        if status != "dead":
            break
        log(f"layout {layout['name']} died mid-run with {len(QRES)} puzzles played; trying the next layout")
    log(f"DONE: {len(QRES)} tasks played, {sum(1 for r in QRES.values() if r.get('verified'))} verified, "
        f"{STATS['tokens'] / 1e6:.2f}M tokens, layouts {STATS.get('layouts_used')}, status {status}")
    if not played:
        log("NO SERVING LAYOUT STARTED - NVARC submission stays")
        sys.exit(3)
    if status == "dead":
        sys.exit(4)  # partial results are merged; the parent may give the GPUs back to NVARC and re-merge


if __name__ == "__main__":
    main()



In [ ]:
%%writefile arckit.py
"""arckit - draft helper toolkit for ARC transform(grid) programs (reference implementation).

25 public functions + the Obj record. Pure numpy/scipy, no I/O, deterministic.
Meant to be exec'd into the sandbox namespace before the model's program, so the
model can call the names directly (no import line needed).

Conventions: grids are numpy int arrays (list-of-lists accepted everywhere); a cell is (r, c);
a direction is (dr, dc); a mask is a bool array; functions never modify their inputs unless
the doc line says "in place".
"""
import numpy as np
from collections import Counter, deque
from scipy import ndimage as _nd

_S4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])
_S8 = np.ones((3, 3), int)
D4 = [(-1, 0), (1, 0), (0, -1), (0, 1)]
D8 = D4 + [(-1, -1), (-1, 1), (1, -1), (1, 1)]


class Obj:
    """One connected object. Fields: cells [(r,c)], r0 c0 r1 c1 (inclusive bbox), h w size,
    mask (h x w bool), full (grid-size bool), patch (h x w ints, non-object cells = bg), color (majority colour),
    colors (Counter), border (touches the grid edge), solid (fills its bbox), n_holes."""

    def __init__(self, g, m, bgc):
        rs, cs = np.nonzero(m)
        self.r0, self.r1, self.c0, self.c1 = int(rs.min()), int(rs.max()), int(cs.min()), int(cs.max())
        self.h, self.w = self.r1 - self.r0 + 1, self.c1 - self.c0 + 1
        self.cells = sorted(zip(rs.tolist(), cs.tolist()))
        self.size = len(self.cells)
        self.full = m                                   # bool mask at full grid size
        self.mask = m[self.r0:self.r1 + 1, self.c0:self.c1 + 1].copy()
        self.patch = np.where(self.mask, g[self.r0:self.r1 + 1, self.c0:self.c1 + 1], bgc)
        self.colors = Counter(g[rs, cs].tolist())
        self.color = self.colors.most_common(1)[0][0]
        self.border = bool(self.r0 == 0 or self.c0 == 0 or self.r1 == g.shape[0] - 1 or self.c1 == g.shape[1] - 1)
        self.solid = bool(self.mask.all())

    @property
    def n_holes(self):
        return len(holes(self.mask))

    @property
    def center(self):
        return ((self.r0 + self.r1) / 2, (self.c0 + self.c1) / 2)

    def __repr__(self):
        return f"Obj(color={self.color}, size={self.size}, box=({self.r0},{self.c0})-({self.r1},{self.c1}))"


# ----------------------------------------------------------------- perception
def bg(g):
    """Background colour = the most frequent colour of the grid."""
    v, n = np.unique(np.asarray(g), return_counts=True)
    return int(v[np.argmax(n)])


def objects(g, bg=None, diag=True, by_color=False):
    """Connected objects of non-background cells, in reading order of their bbox.
    diag=True joins diagonal neighbours; by_color=True makes one object per single-colour component."""
    g = np.asarray(g)
    bgc = globals()['bg'](g) if bg is None else bg
    st = _S8 if diag else _S4
    masks = [g == c for c in np.unique(g) if c != bgc] if by_color else [g != bgc]
    out = []
    for m in masks:
        lab, n = _nd.label(m, structure=st)
        out += [Obj(g, lab == i, bgc) for i in range(1, n + 1)]
    return sorted(out, key=lambda o: (o.r0, o.c0))


def crop(g, o):
    """Sub-grid of the bbox of an Obj, a bool mask, or an (r0, c0, r1, c1) inclusive box."""
    g = np.asarray(g)
    if isinstance(o, Obj):
        r0, c0, r1, c1 = o.r0, o.c0, o.r1, o.c1
    elif isinstance(o, np.ndarray):
        rs, cs = np.nonzero(o); r0, c0, r1, c1 = rs.min(), cs.min(), rs.max(), cs.max()
    else:
        r0, c0, r1, c1 = o
    return g[r0:r1 + 1, c0:c1 + 1].copy()


def holes(mask, diag=False):
    """Enclosed regions: list of bool masks, one per region of ~mask that cannot reach the array edge."""
    mask = np.asarray(mask, bool)
    lab, n = _nd.label(~mask, structure=_S8 if diag else _S4)
    edge = set(lab[0]) | set(lab[-1]) | set(lab[:, 0]) | set(lab[:, -1])
    return [lab == i for i in range(1, n + 1) if i not in edge]


def sep_lines(g, color=None):
    """(rows, cols) that are filled end to end with one colour (optionally a given colour)."""
    g = np.asarray(g)
    ok = lambda v: (v == v[0]).all() and (color is None or v[0] == color)
    return [r for r in range(g.shape[0]) if ok(g[r])], [c for c in range(g.shape[1]) if ok(g[:, c])]


def _spans(n, seps):
    res, s = [], 0
    for x in list(seps) + [n]:
        if x > s:
            res.append((s, x))
        s = x + 1
    return res


def _sep_color(g):
    """Guess the separator colour: among non-background colours with full lines, prefer inner (non-border)
    lines that cut the grid into equal panels; solid content bars at the border or uneven cuts lose."""
    H, W = g.shape
    b = bg(g)
    best, key = None, None
    for c in sorted({int(v) for v in np.unique(g)} - {b}):
        rows, cols = sep_lines(g, c)
        inner = [r for r in rows if 0 < r < H - 1] + [k for k in cols if 0 < k < W - 1]
        if not inner:
            continue
        hs = {y - x for x, y in _spans(H, rows)}
        ws = {y - x for x, y in _spans(W, cols)}
        k = ((len(hs) <= 1) + (len(ws) <= 1), -(len(rows) + len(cols) - len(inner)), len(inner))
        if key is None or k > key:
            best, key = c, k
    return best


def panels(g, color=None):
    """Split the grid at its separator lines: list of rows, each a list of (r0, c0, subgrid).
    Pass the separator colour when you know it; the default guesses it (see _sep_color)."""
    g = np.asarray(g)
    if color is None:
        color = _sep_color(g)
    rows, cols = sep_lines(g, color)
    if color is None and rows and cols:          # no non-background separator: lines of the dominant colour
        c = Counter([int(g[r, 0]) for r in rows] + [int(g[0, k]) for k in cols]).most_common(1)[0][0]
        rows, cols = sep_lines(g, c)
    return [[(a, c, g[a:b, c:d]) for c, d in _spans(g.shape[1], cols)] for a, b in _spans(g.shape[0], rows)]


def d8(a):
    """The 8 orientations of a patch: 4 rotations, then 4 rotations of the left-right mirror."""
    a = np.asarray(a)
    return [np.rot90(a, k) for k in range(4)] + [np.rot90(a[:, ::-1], k) for k in range(4)]


def canon(a, rot=True, flip=True):
    """Hashable key that is equal for two patches/masks/Objs iff they match up to rotation/reflection."""
    a = a.mask if isinstance(a, Obj) else np.asarray(a)
    os_ = d8(a) if (rot and flip) else (d8(a)[:4] if rot else ([a, a[:, ::-1]] if flip else [a]))
    return min((o.shape, o.astype(int).tobytes()) for o in os_)


def same_shape(a, b, rot=True, flip=True, scale=False):
    """True if two masks/Objs have the same shape up to rotation/reflection (and integer scaling if scale)."""
    a = a.mask if isinstance(a, Obj) else np.asarray(a) != 0
    b = b.mask if isinstance(b, Obj) else np.asarray(b) != 0
    def unscale(m):
        for k in range(min(m.shape), 1, -1):
            if m.shape[0] % k == 0 and m.shape[1] % k == 0:
                s = m[::k, ::k]
                if np.array_equal(np.kron(s, np.ones((k, k), bool)), m):
                    return s
        return m
    if scale:
        a, b = unscale(a), unscale(b)
    return canon(a, rot, flip) == canon(b, rot, flip)


def find(g, patch, wildcard=None):
    """All (r, c) where patch occurs in g; patch cells equal to wildcard match anything."""
    g, p = np.asarray(g), np.asarray(patch)
    h, w = p.shape
    care = np.ones_like(p, bool) if wildcard is None else p != wildcard
    return [(r, c) for r in range(g.shape[0] - h + 1) for c in range(g.shape[1] - w + 1)
            if (g[r:r + h, c:c + w][care] == p[care]).all()]


def symmetry_fill(g, unknown):
    """Fill the cells marked unknown (a colour or a bool mask) from their symmetric partners, using every
    mirror axis / diagonal symmetry that the known cells satisfy, and all compositions of those."""
    g = np.asarray(g).copy()
    H, W = g.shape
    unk = (g == unknown) if np.isscalar(unknown) else np.asarray(unknown, bool).copy()
    ii, jj = np.indices((H, W))
    # a symmetry is an affine map (i, j) -> (a*i + b*j + e, c*i + d*j + f)
    cands = [(-1, 0, 0, 1, s, 0) for s in range(H // 2, 2 * H - H // 2)] + [(1, 0, 0, -1, 0, s) for s in range(W // 2, 2 * W - W // 2)]
    cands += [(0, 1, 1, 0, 0, 0)] + [(0, -1, -1, 0, s, s) for s in range(min(H, W) // 2, 2 * max(H, W))]
    def apply(m):
        a, b, c, d, e, f = m
        x, y = a * ii + b * jj + e, c * ii + d * jj + f
        ok = (x >= 0) & (x < H) & (y >= 0) & (y < W)
        return np.where(ok, x, 0), np.where(ok, y, 0), ok
    base, weak = [], []
    for m in cands:
        x, y, ok = apply(m)
        both = ok & ~unk & ~unk[x, y]
        if both.sum() >= 0.3 * H * W:
            rate = (g[both] == g[x, y][both]).mean()
            if rate == 1:
                base.append(m)
            elif rate >= 0.5:
                weak.append((rate, m))
    def closure(gens):
        group = set(gens)
        for _ in range(3):                               # close under composition (bounded)
            for p in list(group):
                for q in gens:
                    if len(group) < 64:
                        a, b, c, d, e, f = p; a2, b2, c2, d2, e2, f2 = q
                        group.add((a2 * a + b2 * c, a2 * b + b2 * d, c2 * a + d2 * c, c2 * b + d2 * d,
                                   a2 * e + b2 * f + e2, c2 * e + d2 * f + f2))
        return sorted(group)
    def run(ms):
        changed = True
        while changed and unk.any():
            changed = False
            for x, y, ok in [apply(m) for m in ms]:
                take = unk & ok & ~unk[x, y]
                if take.any():
                    g[take] = g[x, y][take]; unk[take] = False; changed = True
    run(closure(base))
    if unk.any() and weak:                               # best-effort: add the best approximate symmetry
        run(closure(base + [max(weak)[1]]))
    return g


# ----------------------------------------------------------------- geometry / drawing
def inb(g, r, c):
    """True if (r, c) is inside the grid."""
    return 0 <= r < len(g) and 0 <= c < len(g[0])


def nbrs(g, r, c, diag=False):
    """In-bounds neighbour cells of (r, c): 4 of them, or 8 with diag=True."""
    return [(r + dr, c + dc) for dr, dc in (D8 if diag else D4) if inb(g, r + dr, c + dc)]


def ray(g, start, d, stop=None):
    """Cells visited walking from start (exclusive) in direction d until the grid edge,
    or until stop(colour) is true for the next cell (that cell is not included)."""
    out, (r, c) = [], start
    while True:
        r, c = r + d[0], c + d[1]
        if not inb(g, r, c) or (stop is not None and stop(g[r][c])):
            return out
        out.append((r, c))


def line(g, a, b, color):
    """Draw, in place, the straight (horizontal / vertical / 45-degree) segment from cell a to cell b."""
    n = max(abs(b[0] - a[0]), abs(b[1] - a[1]))
    sr, sc = np.sign(b[0] - a[0]), np.sign(b[1] - a[1])
    for k in range(n + 1):
        r, c = a[0] + sr * k, a[1] + sc * k
        if inb(g, r, c):
            g[r][c] = color
    return g


def paste(g, patch, r, c, transparent=None, color=None):
    """Write patch into g in place with its top-left at (r, c), clipped at the edges. Cells equal to
    transparent are skipped; if patch is a bool mask, its True cells are painted with color."""
    p = np.asarray(patch)
    for i, j in zip(*np.nonzero(p if p.dtype == bool else (np.ones_like(p, bool) if transparent is None else p != transparent))):
        if inb(g, r + i, c + j):
            g[r + i][c + j] = color if p.dtype == bool else p[i, j]
    return g


def slide(g, o, d, bg=None, move=True):
    """Number of steps Obj o can move in direction d before leaving the grid or hitting a non-bg cell;
    with move=True the object is also moved in place."""
    bgc = globals()['bg'](g) if bg is None else bg
    own, k = set(o.cells), 0
    while all(inb(g, r + d[0] * (k + 1), c + d[1] * (k + 1)) and
              ((r + d[0] * (k + 1), c + d[1] * (k + 1)) in own or g[r + d[0] * (k + 1)][c + d[1] * (k + 1)] == bgc)
              for r, c in o.cells):
        k += 1
    if move and k:
        vals = [g[r][c] for r, c in o.cells]
        for r, c in o.cells:
            g[r][c] = bgc
        for (r, c), v in zip(o.cells, vals):
            g[r + d[0] * k][c + d[1] * k] = v
    return k


def flood(passable, start, diag=False):
    """Bool mask of the cells reachable from start (a cell or list of cells) through passable cells."""
    passable = np.asarray(passable, bool)
    lab, _ = _nd.label(passable, structure=_S8 if diag else _S4)
    starts = [start] if isinstance(start[0], (int, np.integer)) else list(start)
    ids = {lab[r, c] for r, c in starts} - {0}
    return np.isin(lab, list(ids))


def dist(passable, sources, diag=False):
    """BFS step distance from the source cells through passable cells (-1 where unreachable).
    sources: a list of (r, c), a single (r, c), or a bool mask."""
    passable = np.asarray(passable, bool)
    d = -np.ones(passable.shape, int)
    q = deque()
    s = np.asarray(sources)
    if s.dtype == bool or (s.ndim == 2 and s.shape == passable.shape and s.shape[1] != 2):
        sources = list(zip(*np.nonzero(s)))
    elif s.ndim == 1 and s.size == 2:
        sources = [tuple(int(x) for x in s)]
    for r, c in sources:
        d[r, c] = 0; q.append((r, c))
    while q:
        r, c = q.popleft()
        for nr, nc in nbrs(passable, r, c, diag):
            if passable[nr, nc] and d[nr, nc] < 0:
                d[nr, nc] = d[r, c] + 1; q.append((nr, nc))
    return d


def outline(mask, diag=True):
    """Bool mask of the cells just outside mask (touching it; diagonal contact counts if diag)."""
    mask = np.asarray(mask, bool)
    return _nd.binary_dilation(mask, structure=_S8 if diag else _S4) & ~mask


def kron(mask, tile, bg=0):
    """Block grid: a copy of tile for every True/non-zero cell of mask, a bg block elsewhere.
    kron(patch != 0, np.ones((k, k))) * colour style scaling is the special case of a 1-colour tile."""
    mask, tile = np.asarray(mask) != 0, np.asarray(tile)
    out = np.full((mask.shape[0] * tile.shape[0], mask.shape[1] * tile.shape[1]), bg, dtype=int)
    for r, c in zip(*np.nonzero(mask)):
        out[r * tile.shape[0]:(r + 1) * tile.shape[0], c * tile.shape[1]:(c + 1) * tile.shape[1]] = tile
    return out


def period(a, axis=1, tol=0):
    """Smallest shift p along axis under which the array repeats (at most tol mismatching cells)."""
    a = np.atleast_2d(np.asarray(a))
    a = a if axis == 1 else a.T
    for p in range(1, a.shape[1]):
        if (a[:, p:] != a[:, :-p]).sum() <= tol:
            return p
    return a.shape[1]


def exact_cover(target, pieces, rot=False, flip=False, use_all=False):
    """Jigsaw solver: place pieces (bool masks or Objs, each at most once) so that they tile the True
    cells of target exactly. Returns [(piece_index, oriented_mask, r, c)] or None."""
    target = np.asarray(target, bool)
    ms = [p.mask if isinstance(p, Obj) else np.asarray(p, bool) for p in pieces]
    def orients(m):
        os_ = d8(m) if (rot and flip) else (d8(m)[:4] if rot else ([m, m[:, ::-1]] if flip else [m]))
        seen, res = set(), []
        for o in os_:
            k = (o.shape, o.tobytes())
            if k not in seen:
                seen.add(k); res.append(o)
        return res
    cand = [orients(m) for m in ms]
    if sum(int(m.sum()) for m in ms) < target.sum() or (use_all and sum(int(m.sum()) for m in ms) != target.sum()):
        return None
    left, used, placed = target.copy(), [False] * len(ms), []
    def rec():
        rem = np.argwhere(left)
        if len(rem) == 0:
            return True
        r0, c0 = rem[0]
        for i in range(len(ms)):
            if used[i]:
                continue
            for o in cand[i]:
                fc = int(np.argmax(o[0]))                       # first True cell of the top row
                top, lft = r0, c0 - fc
                if lft < 0 or top + o.shape[0] > left.shape[0] or lft + o.shape[1] > left.shape[1]:
                    continue
                reg = left[top:top + o.shape[0], lft:lft + o.shape[1]]
                if not reg[o].all():
                    continue
                reg[o] = False; used[i] = True; placed.append((i, o, int(top), int(lft)))
                if rec():
                    return True
                placed.pop(); used[i] = False; reg[o] = True
        return False
    return placed if rec() else None


# ----------------------------------------------------------------- analysis aids (for looking, not for transform)
def describe(g, max_objects=12):
    """Short text summary of a grid: size, colour counts, background, separator lines, objects."""
    g = np.asarray(g)
    b = bg(g)
    rows, cols = sep_lines(g)
    lines = [f"size {g.shape[0]}x{g.shape[1]}  bg {b}  colours {dict(sorted(Counter(g.ravel().tolist()).items()))}"]
    if rows or cols:
        lines.append(f"full lines: rows {rows} cols {cols}")
    obs = objects(g, b)
    lines.append(f"{len(obs)} objects (8-connected, multicolour):")
    for o in obs[:max_objects]:
        lines.append(f"  box ({o.r0},{o.c0})-({o.r1},{o.c1}) {o.h}x{o.w} size {o.size} colours {dict(o.colors)}"
                     f"{' solid' if o.solid else ''}{' holes=' + str(o.n_holes) if o.n_holes else ''}{' edge' if o.border else ''}")
    if len(obs) > max_objects:
        lines.append(f"  ... {len(obs) - max_objects} more")
    return "\n".join(lines)


def diff(a, b):
    """Short text summary of how output b differs from input a (shape change, recoloured cells, where)."""
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        return f"shape {a.shape[0]}x{a.shape[1]} -> {b.shape[0]}x{b.shape[1]} (ratio {b.shape[0] / a.shape[0]:.2f}, {b.shape[1] / a.shape[1]:.2f})"
    ch = a != b
    if not ch.any():
        return "identical"
    rs, cs = np.nonzero(ch)
    pairs = Counter(zip(a[ch].tolist(), b[ch].tolist()))
    return (f"{int(ch.sum())} cells changed in box ({rs.min()},{cs.min()})-({rs.max()},{cs.max()}); "
            f"colour changes {{from->to: n}}: {dict(pairs.most_common(8))}")


PUBLIC = ['bg', 'objects', 'crop', 'holes', 'sep_lines', 'panels', 'd8', 'canon', 'same_shape', 'find', 'symmetry_fill',
          'inb', 'nbrs', 'ray', 'line', 'paste', 'slide', 'flood', 'dist', 'outline', 'kron', 'period', 'exact_cover',
          'describe', 'diff']


In [ ]:
# ---------------------------------------------------------------------------
# Stage A - NVARC primary pass (cheap-first, 4 workers), hard-stopped at NVARC_END.
# ---------------------------------------------------------------------------
import subprocess, sys, time, os, signal
os.environ.update({"UNSLOTH_DISABLE_STATISTICS": "1", "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
                   "OMP_NUM_THREADS": "12", "PYTHONHASHSEED": "0", "ARC_OUT_DIR": "/kaggle/inference_outputs"})
_t = time.time()
_p = subprocess.Popen([sys.executable, "starter.py", "--end-time", f"{NVARC_END}", "--order", "cheap"], start_new_session=True)
while _p.poll() is None and time.time() < NVARC_END + 120:
    time.sleep(10)
if _p.poll() is None:  # a worker still in TTT past the deadline: stop the whole group
    for _sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(_p.pid, _sig)
        except Exception:
            pass
        time.sleep(20)
print(f"stage A rc={_p.poll()} took {(time.time() - _t) / 60:.1f} min")


In [ ]:
# ---------------------------------------------------------------------------
# NVARC selection -> floor submission.json + per-output pool (top grids, vote share) for stage B and the merge.
# ---------------------------------------------------------------------------
import json, hashlib, time
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, hashable
_t = time.time()
TEST_PATH = ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN
             else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
SOL_PATH = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"
data = ArcDataset.from_file(TEST_PATH, keys=VAL_KEYS) if VAL_KEYS else ArcDataset.from_file(TEST_PATH)
if not RERUN:
    data = data.load_replies(SOL_PATH)

def nvarc_select():
    dec = ArcDecoder(data.split_multi_replies(), n_guesses=2)
    dec.load_decoded_results("/kaggle/inference_outputs")
    dec.load_decoded_results("/kaggle/inference_outputs_deep", run_name=".deep")
    selected = dec.run_selection_algo()
    sub = data.get_submission(selected)
    for k in data.keys:
        for i, t in enumerate(data.queries[k]["test"]):
            e = sub[k][i]
            for a in ("attempt_1", "attempt_2"):
                g = e.get(a)
                ok = isinstance(g, list) and len(g) > 0 and all(isinstance(r, list) and len(r) == len(g[0]) and len(r) > 0 for r in g)
                if not ok or g == [[0]]:
                    e[a] = [[int(x) for x in row] for row in t["input"]] if a == "attempt_1" else (g if ok and g != [[0]] else [[0]])
            if e["attempt_2"] == e["attempt_1"]:
                e["attempt_2"] = [[0]]
            e["attempt_1"] = [[int(x) for x in row] for row in e["attempt_1"]]
            e["attempt_2"] = [[int(x) for x in row] for row in e["attempt_2"]]
    pool = {}
    for bk, samples in dec.decoded_results.items():
        votes = {}
        for s in samples.values():
            h = hashable(s["solution"])
            votes[h] = votes.get(h, 0) + 1
        ranked = selected.get(bk, [])
        top = [np.asarray(g).astype(int).tolist() for g in ranked[:3]]
        v = [votes.get(hashable(np.asarray(g)), 0) for g in ranked[:3]]
        pool[bk] = {"top": top, "votes": v, "n": len(samples), "share": (v[0] / len(samples)) if samples and v else 0.0}
    return sub, pool

nvarc_sub, nvarc_pool = nvarc_select()
json.dump(nvarc_sub, open("/kaggle/working/submission.json", "w"))
json.dump(nvarc_sub, open("/kaggle/working/nvarc_floor.json", "w"))
json.dump(nvarc_pool, open("/kaggle/working/nvarc_pool.json", "w"))
n_out = sum(len(data.queries[k]["test"]) for k in data.keys)
n_dec = sum(1 for k in data.keys for i in range(len(data.queries[k]["test"])) if f"{k}_{i}" in nvarc_pool)
detail = f"{n_dec}/{n_out} test outputs decoded, {sum(1 for p in nvarc_pool.values() if p['share'] >= 0.5)} confident (top-1 share >= 50%)"
if not RERUN:
    NVARC_SCORE = 100 * data.validate_submission(nvarc_sub) / len(data.keys)
    DIAG["facts"]["nvarc_only_score"] = NVARC_SCORE
    detail += f"; NVARC-only validation score {NVARC_SCORE:.2f}% on {len(data.keys)} tasks"
level(2, "nvarc-floor", "PASS" if n_dec else "FAIL", detail, T0)


In [ ]:
# ---------------------------------------------------------------------------
# Stage B - Qwen3.8-27B verified program synthesis (adds only). Fallback: NVARC catch-up on unreached tasks.
# ---------------------------------------------------------------------------
import json, os, subprocess, sys, time, signal, glob
_t = time.time()

def gpu_apps():
    try:
        return subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
                              capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception:
        return ""

def wait_gpus_free(limit_s):
    _g = time.time()
    while time.time() - _g < limit_s and gpu_apps():
        time.sleep(5)
    return not gpu_apps()

deadline_wait = min(HARD_END - 1800, time.time() + 2400)
while not os.path.exists("/kaggle/working/vllm_prep.json") and time.time() < deadline_wait and PREP_PROC.poll() is None:
    time.sleep(10)
time.sleep(2)  # prep writes the status file and then warms the page cache; its exit alone is not a failure
try:
    prep = json.load(open("/kaggle/working/vllm_prep.json"))
except Exception as _e:
    prep = {"ok": False, "error": f"prep status unreadable: {_e!r} (prep rc {PREP_PROC.poll()})"}
DIAG["facts"]["vllm_prep"] = {k: v for k, v in prep.items() if k != "pip_tail"}
level(3, "vllm-install", "PASS" if prep.get("ok") else "FAIL",
      f"{prep.get('import', prep.get('error', ''))[:160]} (install {prep.get('pip_s', '?')}s)" if prep.get("ok") else
      f"{prep.get('error', '')} {prep.get('pip_tail', '')[-400:]}", _t)

qwen_dir = None
for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
    try:
        if "Qwen3_5" in open(c).read() and glob.glob(os.path.join(os.path.dirname(c), "*.safetensors")):
            qwen_dir = os.path.dirname(c)
    except Exception:
        pass

# GPUs must be empty before vLLM starts (NVARC workers are separate processes).
DIAG["facts"]["gpus_free_before_qwen"] = wait_gpus_free(300)

QWEN_RC = None
# Serving ladder: each layout must start, report ready and answer a sanity question; a layout that dies mid-run
# hands over to the next one with all results kept. tp4-v1 = the exact flags measured on 2026-10-05.
_COMMON = ["--max-model-len", "65536", "--gpu-memory-utilization", "0.9", "--reasoning-parser", "qwen3",
           "--dtype", "bfloat16", "--language-model-only"]
LAYOUTS = [
    {"name": "tp4-mtp1", "servers": [{"gpus": "0,1,2,3", "port": 1234, "max_seqs": 40, "flags":
        ["--tensor-parallel-size", "4", "--max-num-seqs", "40", "--no-enable-prefix-caching",
         "--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": 1})] + _COMMON}]},
    {"name": "tp4", "servers": [{"gpus": "0,1,2,3", "port": 1235, "max_seqs": 40, "flags":
        ["--tensor-parallel-size", "4", "--max-num-seqs", "40", "--no-enable-prefix-caching"] + _COMMON}]},
    {"name": "tp4-v1", "servers": [{"gpus": "0,1,2,3", "port": 1236, "max_seqs": 24, "flags":
        ["--tensor-parallel-size", "4", "--max-num-seqs", "24", "--enable-prefix-caching"] + _COMMON}]},
]
if prep.get("ok") and qwen_dir and time.time() < HARD_END - 1800:
    cfg = {"test_path": TEST_PATH, "keys": list(data.keys), "nvarc_pool": "/kaggle/working/nvarc_pool.json",
           "nvarc_sub": "/kaggle/working/nvarc_floor.json", "out_sub": "/kaggle/working/submission.json",
           "work_dir": "/kaggle/working", "hard_deadline": HARD_END, "window_s": QWEN_WINDOW_S,
           "python312": prep["python"], "model_dir": qwen_dir, "layouts": LAYOUTS, "max_model_len": 65536,
           "toolkit_path": os.path.abspath("arckit.py"), "effort_first": "low", "effort_second": "low",
           "episode_tokens": 48000, "min_budget": 6000, "first_frac": 0.85, "commit_cap": 3000,
           "repair_cap": 3000, "repair_thinking": False, "max_repairs": 2, "commit_mode": "continue", "per_input_s": 6, "tokens_per_stream": 16000, "temp_thinking": 1.0, "temp_direct": 0.7,
           "prior_stream_rate": 10.0, "second_pass": True, "log_every_s": 300, "start_timeout_s": 1500,
           "min_start_left_s": 1800, "hang_s": 420}
    json.dump(cfg, open("/kaggle/working/qwen_cfg.json", "w"))
    _q = subprocess.Popen([sys.executable, "qwen_stage.py", "/kaggle/working/qwen_cfg.json"], start_new_session=True)
    while _q.poll() is None and time.time() < HARD_END + 300:
        time.sleep(10)
    if _q.poll() is None:
        for _sig in (signal.SIGTERM, signal.SIGKILL):
            try:
                os.killpg(_q.pid, _sig)
            except Exception:
                pass
            time.sleep(15)
    QWEN_RC = _q.poll()
print(f"stage B rc={QWEN_RC} took {(time.time() - _t) / 60:.1f} min")

if QWEN_RC != 0 and time.time() < HARD_END - 1500:
    # Fallback (no layout started, or every layout died mid-run): give the GPUs back to NVARC for the tasks it
    # never reached, re-select, and re-merge whatever Qwen already verified (adds only).
    print(f"[fallback] Qwen stage rc={QWEN_RC} - NVARC catch-up on unreached tasks")
    # vLLM servers run in their own sessions: if the child died abnormally they may still hold the GPUs.
    for _pat in ("vllm.entrypoints", "/tmp/py312/python/bin/python3.12"):
        try:
            subprocess.run(["pkill", "-9", "-f", _pat], capture_output=True, timeout=30)
        except Exception as _e:
            print("[fallback] pkill:", repr(_e))
    print(f"[fallback] GPUs free: {wait_gpus_free(180)}")
    from starter import estimated_work
    unreached = [k for k in data.keys if not any(f"{k}_{i}" in nvarc_pool for i in range(len(data.queries[k]["test"])))]
    unreached = sorted(unreached, key=lambda k: estimated_work(data.queries[k]))
    if unreached:
        json.dump(unreached, open("/kaggle/working/keys_catchup.json", "w"))
        env = dict(os.environ, ARC_RESUME="1")
        _c = subprocess.Popen([sys.executable, "starter.py", "--end-time", f"{HARD_END - 600}", "--keys-file",
                               "/kaggle/working/keys_catchup.json", "--order", "file"], env=env, start_new_session=True)
        while _c.poll() is None and time.time() < HARD_END:
            time.sleep(10)
        if _c.poll() is None:
            try:
                os.killpg(_c.pid, signal.SIGKILL)
            except Exception:
                pass
        nvarc_sub, nvarc_pool = nvarc_select()
        json.dump(nvarc_sub, open("/kaggle/working/nvarc_floor.json", "w"))
        json.dump(nvarc_pool, open("/kaggle/working/nvarc_pool.json", "w"))
        _final = nvarc_sub
        if os.path.exists("/kaggle/working/qwen_results.json"):
            import qwen_stage as _qs
            _qr = json.load(open("/kaggle/working/qwen_results.json")).get("results", {})
            _m, _how = _qs.merge(nvarc_sub, nvarc_pool, _qr)
            if _qs.check_invariant(_m, nvarc_pool):
                _final = _m
                print(f"[fallback] re-merged {len(_qr)} Qwen results: {_how}")
        json.dump(_final, open("/kaggle/working/submission.json", "w"))
        print(f"[fallback] re-selected after catch-up of {len(unreached)} tasks")


In [ ]:
# ---------------------------------------------------------------------------
# Final check, scoring (validation) and the diagnostic game board.
# ---------------------------------------------------------------------------
import json, os
# Repair instead of assert: an exception here would void the whole submission, while a repaired row
# (input grid as the guess) only loses that one output.
try:
    sub = json.load(open("/kaggle/working/submission.json"))
except Exception as _e:
    print("submission.json unreadable:", repr(_e)); sub = {}
if not isinstance(sub, dict):
    sub = {}
def _ok(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and isinstance(g[0], list) and 0 < len(g[0]) <= 30 and
            all(isinstance(r, list) and len(r) == len(g[0]) and all(isinstance(v, int) and 0 <= v <= 9 for v in r) for r in g))
_challenges_final = json.load(open(_sn_path))
if not RERUN:  # validation scores only the VAL_KEYS
    _challenges_final = {k: _challenges_final[k] for k in data.keys}
try:
    _floor_final = json.load(open("/kaggle/working/nvarc_floor.json"))
except Exception:
    _floor_final = {}
_repaired = 0
for k, t in _challenges_final.items():
    rows = sub.get(k) if isinstance(sub.get(k), list) else []
    frows = _floor_final.get(k) if isinstance(_floor_final.get(k), list) else []
    fixed = []
    for i, x in enumerate(t["test"]):
        e = rows[i] if i < len(rows) and isinstance(rows[i], dict) else {}
        a1, a2 = e.get("attempt_1"), e.get("attempt_2")
        if not (_ok(a1) and _ok(a2)):
            _repaired += 1
            f = frows[i] if i < len(frows) and isinstance(frows[i], dict) else {}
            s1, s2 = _sn_attempts(x["input"])
            a1 = a1 if _ok(a1) else (f.get("attempt_1") if _ok(f.get("attempt_1")) else s1)
            a2 = a2 if _ok(a2) else (f.get("attempt_2") if _ok(f.get("attempt_2")) else s2)
        fixed.append({"attempt_1": a1, "attempt_2": a2})
    sub[k] = fixed
sub = {k: sub[k] for k in _challenges_final}
json.dump(sub, open("/kaggle/working/submission.json", "w"))
print(f"[final] submission.json: {len(sub)} tasks, {_repaired} rows repaired")
DIAG["facts"]["rows_repaired"] = _repaired
qr = json.load(open("/kaggle/working/qwen_results.json")) if os.path.exists("/kaggle/working/qwen_results.json") else {}
stats, res = qr.get("stats", {}), qr.get("results", {})
level(4, "qwen-game", "PASS" if res and QWEN_RC == 0 else ("WARN" if res else ("SKIP" if QWEN_RC is None else "FAIL")),
      f"rc {QWEN_RC}, layouts {stats.get('layouts_used')} ready at {stats.get('layout_ready_s')}s, episodes {stats.get('episodes', 0)} (2nd pass "
      f"{stats.get('second_pass', 0)}), puzzles {len(res)}, verified {sum(1 for r in res.values() if r.get('verified'))}, "
      f"tokens {stats.get('tokens', 0) / 1e6:.2f}M, commits {stats.get('commits', 0)} (continue {stats.get('commit_continue', 0)}, "
      f"chat {stats.get('commit_commit', 0)}, continue-fail {stats.get('continue_fail', 0)}), no-time {stats.get('no_time', 0)}, "
      f"toolkit-error {bool(stats.get('toolkit_error'))}, server-down {stats.get('server_down', 0)}, "
      f"merge {qr.get('merge')}, servers {[(s.get('port'), s.get('conc'), s.get('kv_capacity')) for s in qr.get('servers', [])]}", T0)
DIAG["facts"]["qwen_stats"] = stats

if not RERUN:
    sols = json.load(open(SOL_PATH))
    def task_score(s, k):
        return sum(any(e[a] == sols[k][i] for a in ("attempt_1", "attempt_2")) for i, e in enumerate(s[k])) / len(sols[k])
    floor = json.load(open("/kaggle/working/nvarc_floor.json"))
    rows, tot_h, tot_n, ver_ok, ver_all = [], 0.0, 0.0, 0, 0
    print("\n" + "=" * 110 + "\n PUZZLE BOARD  (N = NVARC floor, Q = Qwen program, H = hybrid; X = ARC Prize xhigh solved it)\n" + "=" * 110)
    for k in data.keys:
        n_s, h_s = task_score(floor, k), task_score(sub, k)
        q = res.get(k, {})
        qok = None
        if q.get("verified"):
            ver_all += 1
            qok = all(q["tests"][i] == sols[k][i] for i in range(len(sols[k])))
            ver_ok += int(qok)
        share = max((nvarc_pool.get(f"{k}_{i}", {}).get("share", 0) for i in range(len(sols[k]))), default=0)
        tot_n += n_s; tot_h += h_s
        print(f" {k}  N {n_s:.2f} (share {share:.2f})  Q {'VERIFIED ' + ('right' if qok else 'WRONG') if q.get('verified') else ('demos ' + str(q.get('passed', '-')) if q else 'not played'):<18}"
              f"  H {h_s:.2f}  {'X' if k in QWEN_XHIGH_SOLVED else ' '}  {' > '.join(q.get('steps', []))[:60]}")
    K = len(data.keys)
    import math
    p = tot_h / K
    z = 1.96; d = 1 + z * z / K; c = (p + z * z / (2 * K)) / d; hw = z * math.sqrt(p * (1 - p) / K + z * z / (4 * K * K)) / d
    xs = [k for k in data.keys if k in QWEN_XHIGH_SOLVED]
    q_hat = (sum(1 for k in xs if res.get(k, {}).get("verified") and all(res[k]["tests"][i] == sols[k][i] for i in range(len(sols[k])))) / len(xs)) if xs else None
    DIAG["facts"].update({"hybrid_score": 100 * p, "ci95": [100 * max(0, c - hw), 100 * min(1, c + hw)], "nvarc_score": 100 * tot_n / K,
                          "verified_precision": (ver_ok / ver_all) if ver_all else None, "q_hat": q_hat})
    level(5, "score", "PASS",
          f"HYBRID {100 * p:.2f}% (95% CI {100 * max(0, c - hw):.0f}-{100 * min(1, c + hw):.0f}) vs NVARC-only {100 * tot_n / K:.2f}% on {K} "
          f"public-eval tasks at rerun-equal time per task; verified programs {ver_all} ({ver_ok} right); q_hat {q_hat}", T0)
print("\n" + "=" * 110 + "\n DIAGNOSTIC GAME\n" + "=" * 110)
for r in DIAG["levels"]:
    print(f" L{r['level']} {r['name']:<14} {r['status']:<5} {r['s']:>8.0f}s  {str(r['detail'])[:170]}")
for line in qr.get("game_log", [])[-60:]:
    print("  ", line)
json.dump(DIAG, open("/kaggle/working/diag_report.json", "w"), indent=1, default=str)
print("done")
